# KYC identity extraction — Qwen VL OCR pipeline

Complete, runnable, fully offline. Set `MODEL_PATH`, `ZIP_PATH` and `OUTPUT_DIR` in **CELL 3**,
then run top to bottom.

| Cell | Contents | Cell | Contents |
|---|---|---|---|
| 1 | Environment / versions | 11 | MRZ detection and cropping |
| 2 | Imports and dependency check | 12 | Qwen inference |
| 3 | Configuration | 13 | JSON parsing and validation |
| 4 | GPU diagnostics | 14 | Page processing |
| 5 | Model + processor (loaded ONCE) | 15 | Customer / PDF processing |
| 6 | ZIP extraction | 16 | Batch processing |
| 7 | Customer / document discovery | 17 | Performance logging |
| 8 | Document presence report | 18 | Output generation |
| 9 | PDF rendering | 19 | Benchmark summary |
| 10 | Preprocessing / orientation | 20 | Result inspection |

---

## Why page 2 / the MRZ fails — the actual mechanism

Not a prompt problem. A pixel problem, and it is arithmetic:

ICAO 9303 OCR-B characters are **3.2 mm** tall. A passport data page scanned on A4 at 300 dpi is
3509 px tall. Resize that to `MAX_IMAGE_DIMENSION = 1600` and the scale factor is 0.456, so each
MRZ character arrives at the model **17 px tall and 15 px wide** — and that is the *optimistic*
case, where the passport fills the page. If the passport occupies half the frame, or the scan is
a phone photo with margins, characters land at 8–10 px. At that size `0`/`O`, `1`/`I`, `5`/`S`,
`8`/`B` are genuinely indistinguishable, and a correctly-built anti-hallucination prompt does
exactly what it was told to do: it returns `null`.

**The trap:** raising `PDF_RENDER_DPI` does *nothing* if the page is then resized to a fixed
maximum dimension. Final character size depends only on `MAX_IMAGE_DIMENSION` and how much of the
frame the document occupies. Rendering at 600 dpi and downscaling to 1600 px gives byte-for-byte
the same image as rendering at 200 dpi and downscaling to 1600 px.

```
render 300 dpi -> max_dim 1600 : MRZ char 17.2 px tall   (marginal)
render 600 dpi -> max_dim 1600 : MRZ char 17.2 px tall   (identical - the DPI was thrown away)
render 300 dpi -> max_dim 1024 : MRZ char 11.0 px tall   (fails)
```

### The fix: re-render the MRZ band from the PDF, do not upscale pixels

Once the MRZ band is localised, its rectangle is mapped back into **PDF page coordinates** and
that region is re-rendered from the vector source with PyMuPDF's `clip` at `MRZ_RENDER_DPI`.
This is real resolution recovered from the document, not interpolation inventing detail:

| Path | MRZ char height | Visual tokens |
|---|---|---|
| Full page at `max_dim=1600` | 17.2 px | ~2 308 |
| **Dedicated MRZ clip @ 600 dpi** | **41.0 px** | **~365** |

2.4× the character resolution at **one sixth** the token cost. Better accuracy *and* faster — the
MRZ pass is cheaper than the full-page pass it rescues. When the page was deskewed (so the inverse
transform to PDF space is not exact) the pipeline falls back to Lanczos upscaling of the rendered
crop and records `mrz_source="pixel_upscale"`, because the two are not equally trustworthy and the
log should say which one produced the value.

## Why the previous implementation took ~6 minutes

| Item | Cost per 5-page document | Share |
|---|---|---|
| **Decode**: `max_new_tokens=1024`, no stop condition, thinking mode on | 150–200 s | **~60%** |
| Full-page retry on weak pages | 60 s | 20% |
| Enhancement at full 8.7 MP *before* downscaling | 10 s | 3% |
| VLM call just to detect orientation (full prefill for 4 output tokens) | 8 s | 2% |
| Vision encoder + prefill | 4 s | 1% |
| PDF render | 2 s | 1% |

Decoding dominates. A 27B model in bf16 on an H100 decodes at ~20–35 tok/s (54 GB of weights,
HBM-bandwidth bound), so every unnecessary output token costs ~35 ms. What changed:

1. Thinking mode disabled, `MAX_NEW_TOKENS=320`, brace-balance stop, `{` prefilled — the model
   stops the moment the JSON closes instead of decoding to the cap.
2. Orientation by computer vision (~20 ms) instead of a VLM call (~1.5 s).
3. Targeted retries: only failed fields, only failed pages, ~96 output tokens.
4. Preprocessing after downscaling, and only where the measured quality justifies it.
5. The system prompt is ~450 tokens instead of ~1 200, and it is re-prefilled on every call.

**Accuracy is not traded for speed.** The MRZ gets *more* pixels than before, not fewer.

In [ ]:
# =========================================================================
# CELL 1 — ENVIRONMENT / VERSION INFORMATION
# =========================================================================
# Printed first so that any compatibility problem is visible before anything expensive runs.
import os, sys, platform, subprocess, importlib, json

# These must be set BEFORE torch/transformers are imported anywhere in the process.
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # never reach the Hub
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# expandable_segments turns "plenty free but still OOM" (fragmentation) into a working run.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B/main"

print(f"{'python':<16}: {platform.python_version()}")
print(f"{'platform':<16}: {platform.platform()}")

_versions = {}
for _mod in ("torch", "transformers", "accelerate", "numpy", "pandas", "PIL", "cv2",
             "pymupdf", "fitz", "qwen_vl_utils", "flash_attn"):
    try:
        m = importlib.import_module(_mod)
        _versions[_mod] = getattr(m, "__version__", "present")
    except Exception:
        _versions[_mod] = None

for k, v in _versions.items():
    print(f"{k:<16}: {v if v else 'NOT INSTALLED'}")

try:
    import torch
    print(f"{'cuda (torch)':<16}: {torch.version.cuda} | available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"{'gpu[' + str(i) + ']':<16}: {p.name} | {p.total_memory/2**30:.1f} GiB | "
                  f"sm_{p.major}{p.minor}")
except Exception as exc:
    print("torch unavailable:", exc)

try:
    print(f"{'nvidia-smi':<16}: " + subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=15).stdout.strip())
except Exception:
    pass

print(f"{'model path':<16}: {MODEL_PATH}")
print(f"{'model exists':<16}: {os.path.isdir(MODEL_PATH)}")
_cfg_file = os.path.join(MODEL_PATH, "config.json")
if os.path.isfile(_cfg_file):
    with open(_cfg_file) as fh:
        _raw = json.load(fh)
    print(f"{'model_type':<16}: {_raw.get('model_type')}")
    print(f"{'architectures':<16}: {_raw.get('architectures')}")
    print(f"{'vision tower':<16}: {'yes' if 'vision_config' in _raw else 'NO -- see CELL 5'}")
else:
    print("!! config.json not found at MODEL_PATH; CELL 5 will fail")

In [ ]:
# =========================================================================
# CELL 2 — IMPORTS AND DEPENDENCY CHECK
# =========================================================================
# Every optional dependency is guarded. The notebook degrades instead of crashing, and says
# clearly what is missing (section 31: no blind pip installs, no internet).
from __future__ import annotations

import io, re, gc, csv, math, time, zipfile, hashlib, difflib, logging, unicodedata, traceback
from collections import OrderedDict, defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone, date
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

Image.MAX_IMAGE_PIXELS = None          # 300+ dpi A4 scans exceed Pillow's default guard

REQUIRED = {"numpy": np, "pandas": pd, "PIL": Image}

# --- PyMuPDF: mandatory. Page-by-page rendering AND the MRZ clip re-render depend on it. ---
try:
    import pymupdf as fitz
except Exception:
    try:
        import fitz
    except Exception:
        fitz = None

# --- OpenCV: strongly recommended. Orientation, quality metrics and MRZ localisation. ---
try:
    import cv2
except Exception:
    cv2 = None

# --- Tesseract: optional, used ONLY for page orientation (never for transcription). ---
try:
    import pytesseract
    from pytesseract import Output as TessOutput
    pytesseract.get_tesseract_version()
except Exception:
    pytesseract, TessOutput = None, None

try:
    import torch
except Exception:
    torch = None

_missing = []
if fitz is None:
    _missing.append("pymupdf  (MANDATORY: PDF page rendering and MRZ clip re-render)")
if cv2 is None:
    _missing.append("opencv-python-headless  (MRZ localisation and orientation are disabled "
                    "without it; the pipeline still runs, with reduced MRZ recall)")
if torch is None:
    _missing.append("torch  (MANDATORY for inference; mock mode still works)")
if _missing:
    print("MISSING DEPENDENCIES")
    for m in _missing:
        print("  -", m)
else:
    print("all required dependencies present")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
utcnow = lambda: datetime.now(timezone.utc).isoformat()
print("run id:", RUN_ID)

In [ ]:
# =========================================================================
# CELL 3 — CONFIGURATION  (the only cell you normally need to edit)
# =========================================================================
@dataclass
class Config:
    # ---------------- paths -----------------------------------------------
    model_path: str = MODEL_PATH
    zip_path: Path = Path("/domino/datasets/local/kyc/kyc_documents.zip")
    output_dir: Path = Path("/domino/datasets/local/kyc/output")
    # Verification reference ONLY. Never an OCR source, never used to complete a value.
    CUSTOMER_DATABASE_PATH: Optional[Path] = Path("/domino/datasets/local/kyc/customers.csv")

    # ---------------- PDF rendering ---------------------------------------
    # Render high, then decide per region what to downscale. NOTE: raising this alone does NOT
    # improve MRZ legibility -- the final character size is set by MAX_IMAGE_DIMENSION.
    PDF_RENDER_DPI: int = 300
    # Longest side of the full-page image sent to the model. Qwen-VL spends roughly one visual
    # token per 28x28 px block, so 1280 px ~= 1480 tokens. This is the accuracy/latency dial for
    # the FULL PAGE only; small regions get their own treatment below.
    MAX_IMAGE_DIMENSION: int = 1400
    MAX_VISUAL_TOKENS: int = 1600          # hard cap handed to the processor as max_pixels
    MAX_PAGES_PER_PDF: int = 12

    # ---------------- MRZ pipeline ----------------------------------------
    # The MRZ band is re-rendered from the PDF at this DPI (true resolution, not interpolation).
    MRZ_RENDER_DPI: int = 600
    # Used only when the clip cannot be mapped back to PDF space (e.g. after deskew): Lanczos
    # upscaling of the rendered crop. Recorded separately because it is less trustworthy.
    MRZ_UPSCALE_FACTOR: float = 3.0
    MRZ_MAX_WIDTH: int = 1600              # a 1600x180 strip costs only ~365 visual tokens
    MRZ_MIN_CHAR_HEIGHT_PX: float = 22.0   # below this, escalate the MRZ retry ladder
    MRZ_BAND_PAD: float = 0.035            # fraction of page size added around the detected band
    ENABLE_MRZ_THRESHOLD_VARIANT: bool = True   # last-resort binarised variant on retry

    # ---------------- image preprocessing (applied ONLY when measured as needed) -----
    ENABLE_DESKEW: bool = True
    ENABLE_CONTRAST_ENHANCEMENT: bool = True    # CLAHE, only when contrast is measurably low
    ENABLE_SHARPENING: bool = True              # mild unsharp mask, MRZ CROPS ONLY (see CELL 11)
    ENABLE_DENOISE: bool = True                 # edge-preserving only
    ENABLE_ORIENTATION_CORRECTION: bool = True
    DESKEW_MIN_DEG: float = 0.4
    DESKEW_MAX_DEG: float = 15.0
    BLUR_VAR_POOR: float = 80.0            # variance of Laplacian, measured at a fixed 1000 px
    BLUR_VAR_GOOD: float = 300.0
    CONTRAST_LOW: float = 0.30             # (p95-p5)/255
    NOISE_SIGMA_HIGH: float = 6.0
    ILLUMINATION_POOR: float = 0.14
    INK_RATIO_BLANK: float = 0.002

    # ---------------- inference -------------------------------------------
    TORCH_DTYPE: str = "bfloat16"          # H100 native; do NOT use fp16 for a bf16 checkpoint
    DEVICE_MAP: str = "auto"
    ATTN_IMPL: Optional[str] = None        # None = autodetect flash_attention_2 -> sdpa -> eager
    MAX_NEW_TOKENS: int = 320              # sized to the schema: 12 fields x ~20 tokens + braces
    MAX_NEW_TOKENS_TARGETED: int = 96      # field-level retry asks for less, so it decodes less
    MAX_NEW_TOKENS_MRZ: int = 128          # 2-3 lines of <=44 chars
    TEMPERATURE: float = 0.0               # deterministic: sampling is how characters get invented
    REPETITION_PENALTY: float = 1.0        # MUST be 1.0 -- see the warning in CELL 12
    NO_REPEAT_NGRAM_SIZE: int = 0          # MUST be 0 -- it would forbid "<<" in an MRZ
    DISABLE_THINKING: bool = True          # Qwen3 templates enable it by default; pure cost here
    RESERVE_VRAM_GIB: float = 8.0          # headroom device_map is not allowed to fill with weights
    CPU_OFFLOAD_GIB: float = 64.0
    OOM_DOWNSCALE_FACTOR: float = 0.65
    OOM_MAX_DOWNSCALES: int = 2
    MOCK_MODEL: bool = False               # True = rehearse the pipeline with no GPU

    # ---------------- secondary documents / handwriting -------------------
    PROCESS_SECONDARY_DOCUMENTS: bool = True
    VERIFY_DOCUMENT_TITLES: bool = True
    TITLE_BAND_FRACTION: float = 0.34      # top fraction of the page that carries the title
    TITLE_RENDER_DPI: int = 400            # the title band is re-rendered, like the MRZ band
    HANDWRITING_TILES: int = 2             # halves of the page, each at full MAX_IMAGE_DIMENSION
    HANDWRITING_RENDER_DPI: int = 450      # handwriting needs stroke detail, not just size
    MAX_HANDWRITING_ATTEMPTS: int = 2
    TITLE_MATCH_THRESHOLD: float = 0.72    # difflib ratio on normalised titles; score is exported

    # ---------------- name matching (STAGE B only) ------------------------
    # Normalisation and fuzzy scores exist ONLY for comparison. They never touch a raw value.
    NAME_FUZZY_THRESHOLD: float = 0.90     # below this a difference is a MISMATCH, not a match
    EXPOSE_FUZZY_SCORES: bool = True

    # ---------------- retry policy ----------------------------------------
    MAX_FIELD_RETRIES: int = 1             # targeted retry of missing fields, per page
    MAX_MRZ_ATTEMPTS: int = 3              # MRZ ladder: clip -> upscale+sharpen -> threshold
    RETRY_ONLY_MISSING_FIELDS: bool = True
    CRITICAL_FIELDS: Tuple[str, ...] = ("surname", "given_names", "date_of_birth",
                                        "document_number")

    # ---------------- run control / outputs -------------------------------
    SAVE_DEBUG_IMAGES: bool = False        # True = also keep images for SUCCESSFUL pages
    SAVE_DEBUG_ON_FAILURE: bool = True     # failed/low-confidence pages and MRZ crops
    RESUME: bool = True
    LIMIT_CUSTOMERS: Optional[int] = None  # None = all; set an int for a smoke test
    MAX_CONSECUTIVE_ERRORS: int = 3        # circuit breaker: stop if the GPU is unusable

    def dirs(self) -> Dict[str, Path]:
        d = {k: self.output_dir / v for k, v in {
            "pdfs": "00_selected_pdfs", "results": "01_results", "reports": "02_reports",
            "debug": "03_debug", "logs": "04_logs"}.items()}
        for p in d.values():
            p.mkdir(parents=True, exist_ok=True)
        return d


CFG = Config()
DIRS = CFG.dirs()

# Canonical document names. Column keys for the presence report follow section 4 exactly.
IDENTITY_DOC = "JUSTIFICATIF IDENTITE.PDF"
DOMICILE_DOC = "JUSTIFICATIF DOMICILE.PDF"
CONVENTION_DOC = "CONVENTION COMPTE.PDF"
FATCA_DOC = "FATCA.PDF"
SIGNATURE_DOC = "CARTON SIGNATURE.PDF"         # earlier specs spelled it SIGNATUTE; both match
DOCUMENTS_REQUIRED = [IDENTITY_DOC, DOMICILE_DOC, CONVENTION_DOC, FATCA_DOC, SIGNATURE_DOC]
PRESENCE_COLUMNS = OrderedDict([
    (IDENTITY_DOC, "identity_document_exists"),
    (DOMICILE_DOC, "domicile_document_exists"),
    (CONVENTION_DOC, "convention_compte_exists"),
    (FATCA_DOC, "fatca_exists"),
    (SIGNATURE_DOC, "signature_card_exists"),
])

# Identity fields (section 9). surname_latin / given_names_latin are mandatory and are read ONLY
# from Latin characters actually visible on the identity document.
IDENTITY_FIELDS = ["document_type", "surname_latin", "given_names_latin", "date_of_birth",
                   "place_of_birth", "nationality", "sex", "document_number", "issue_date",
                   "expiry_date", "issuing_authority", "personal_number", "mrz"]
NAME_FIELDS = ["surname_latin", "given_names_latin"]

print("model   :", CFG.model_path)
print("zip     :", CFG.zip_path, "| exists:", CFG.zip_path.exists())
print("output  :", CFG.output_dir)
for k, v in DIRS.items():
    print(f"  {k:<8}: {v}")

In [ ]:
# =========================================================================
# CELL 4 — GPU DIAGNOSTICS
# =========================================================================
def gpu_memory() -> Dict[str, float]:
    """Per-device totals in GiB as seen by THIS process."""
    if torch is None or not torch.cuda.is_available():
        return {}
    out = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out[f"gpu{i}_total_gib"] = round(total / 2**30, 1)
        out[f"gpu{i}_free_gib"] = round(free / 2**30, 1)
        out[f"gpu{i}_allocated_gib"] = round(torch.cuda.memory_allocated(i) / 2**30, 1)
    return out


def gpu_report() -> None:
    """Memory for this process AND any other process on the card.

    A second PID here is the most common explanation for 'OOM on every call'."""
    if torch is None or not torch.cuda.is_available():
        print("no CUDA device visible -- set CFG.MOCK_MODEL = True to rehearse the pipeline")
        return
    print(f"device            : {torch.cuda.get_device_name(0)}")
    for k, v in gpu_memory().items():
        print(f"{k:<18}: {v} GiB")
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
                              "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=15).stdout.strip()
        print("processes on GPU  :", out or "(none)")
        print("this pid          :", os.getpid())
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


def detect_attention_impl(cfg: Config = CFG) -> str:
    """Pick the best attention kernel actually available. Never assume flash-attn is installed.

    FA2 needs: the package, an Ampere-or-newer GPU (H100 = sm_90), and a 16-bit dtype."""
    if cfg.ATTN_IMPL:
        return cfg.ATTN_IMPL
    if torch is None or not torch.cuda.is_available():
        return "eager"
    try:
        import flash_attn  # noqa: F401
        major = torch.cuda.get_device_capability(0)[0]
        if major >= 8 and cfg.TORCH_DTYPE in ("bfloat16", "float16"):
            log.info("flash_attn detected and compatible (sm_%d0) -> flash_attention_2", major)
            return "flash_attention_2"
        log.info("flash_attn present but incompatible (sm_%d0 / %s) -> sdpa", major, cfg.TORCH_DTYPE)
    except Exception:
        log.info("flash_attn not installed -> sdpa (PyTorch fused attention)")
    return "sdpa"


def release_cuda_cache() -> None:
    """Deliberately NOT called per page: emptying the cache forces re-allocation and is slow.
    Reserved for OOM recovery and teardown (section 13)."""
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


gpu_report()
ATTN_IMPL = detect_attention_impl(CFG)
print("attention impl    :", ATTN_IMPL)

In [ ]:
# =========================================================================
# CELL 5 — MODEL AND PROCESSOR INITIALISATION  (exactly once per kernel)
# =========================================================================
# The model class is chosen by INSPECTING the local config.json, never assumed. Order:
#   1. the exact class named in config["architectures"], if transformers exposes it
#   2. AutoModelForImageTextToText  (the modern multimodal entry point)
#   3. AutoModelForVision2Seq       (older transformers)
# A checkpoint with no vision tower is rejected outright rather than silently producing
# text-only guesses about an image it never saw.
VISION_HINTS = ("vl", "vision", "image_text", "qwen2_vl", "qwen2_5_vl", "qwen3_vl")


class QwenOCREngine:
    """Holds the model and processor. Both are created once and reused for every page."""
    _instance: Optional["QwenOCREngine"] = None

    def __init__(self, cfg: Config = CFG):
        from transformers import AutoConfig, AutoProcessor
        import transformers as tf
        self.cfg = cfg
        assert torch is not None, "PyTorch is required"
        assert Path(cfg.model_path).exists(), f"model path not found: {cfg.model_path}"

        before = gpu_memory()
        busy = max((v for k, v in before.items() if k.endswith("allocated_gib")), default=0.0)
        if busy > 1.0:
            log.warning("%.1f GiB already allocated on this GPU by this process. If you re-ran "
                        "this cell, call free_model() or restart the kernel: two copies of the "
                        "weights will OOM on every generate().", busy)

        self.hf_config = AutoConfig.from_pretrained(cfg.model_path, trust_remote_code=True,
                                                    local_files_only=True)
        archs = list(getattr(self.hf_config, "architectures", []) or [])
        mtype = str(getattr(self.hf_config, "model_type", "")).lower()
        self.is_vision = hasattr(self.hf_config, "vision_config") or \
            any(h in " ".join(archs).lower() or h in mtype for h in VISION_HINTS)
        if not self.is_vision:
            raise RuntimeError(
                f"{cfg.model_path}\n  model_type={mtype!r} architectures={archs}\n"
                "  This checkpoint exposes no vision tower, so it cannot read an image. Point "
                "CFG.model_path at the VL checkpoint, or set CFG.MOCK_MODEL=True to rehearse.")

        # ---- processor: ONE instance, with the visual-token budget baked in -------
        px = cfg.MAX_VISUAL_TOKENS * 28 * 28      # Qwen-VL: ~1 visual token per 28x28 px block
        try:
            self.processor = AutoProcessor.from_pretrained(
                cfg.model_path, trust_remote_code=True, local_files_only=True,
                min_pixels=256 * 28 * 28, max_pixels=px)
        except TypeError:      # older processors do not accept the pixel bounds
            self.processor = AutoProcessor.from_pretrained(cfg.model_path, trust_remote_code=True,
                                                           local_files_only=True)
        self.tokenizer = getattr(self.processor, "tokenizer", self.processor)
        try:
            self.tokenizer.padding_side = "left"   # required for correct batched generation
        except Exception:
            pass
        self.has_chat_template = bool(getattr(self.processor, "chat_template", None) or
                                      getattr(self.tokenizer, "chat_template", None))

        kwargs: Dict[str, Any] = dict(
            torch_dtype=getattr(torch, cfg.TORCH_DTYPE), device_map=cfg.DEVICE_MAP,
            trust_remote_code=True, local_files_only=True, low_cpu_mem_usage=True,
            attn_implementation=ATTN_IMPL)

        # Reserve headroom: without a budget, accelerate fills the card with WEIGHTS and leaves
        # nothing for the vision encoder, KV cache and logits -- the classic "weights load fine,
        # every generate() OOMs". With a budget the overflow spills to CPU: slower, but it runs.
        if cfg.RESERVE_VRAM_GIB > 0 and torch.cuda.is_available() and cfg.DEVICE_MAP == "auto":
            mm: Dict[Any, str] = {}
            for i in range(torch.cuda.device_count()):
                _, total = torch.cuda.mem_get_info(i)
                mm[i] = f"{max(1, int(total / 2**30) - int(cfg.RESERVE_VRAM_GIB))}GiB"
            mm["cpu"] = f"{int(cfg.CPU_OFFLOAD_GIB)}GiB"
            kwargs["max_memory"] = mm
            log.info("max_memory budget %s (reserving %.0f GiB for activations)",
                     mm, cfg.RESERVE_VRAM_GIB)

        last, self.model = None, None
        for cls_name in archs + ["AutoModelForImageTextToText", "AutoModelForVision2Seq"]:
            if not hasattr(tf, cls_name):
                continue
            try:
                t0 = time.perf_counter()
                self.model = getattr(tf, cls_name).from_pretrained(cfg.model_path, **kwargs)
                self.loader, self.load_s = cls_name, round(time.perf_counter() - t0, 1)
                break
            except Exception as exc:
                last = f"{cls_name}: {type(exc).__name__}: {exc}"
                log.warning("loader %s failed: %s", cls_name, type(exc).__name__)
                release_cuda_cache()
        if self.model is None:
            raise RuntimeError(f"could not load the model. last error: {last}")

        self.model.eval()
        gen_cfg = getattr(self.model, "generation_config", None)
        if gen_cfg is not None:        # neutralise any sampling defaults shipped in the checkpoint
            gen_cfg.do_sample = False
            gen_cfg.temperature = gen_cfg.top_p = gen_cfg.top_k = None
        self.name = f"{Path(cfg.model_path).parent.name} [{self.loader}]"
        self.consecutive_ooms = 0
        log.info("model ready: %s in %.1fs | chat_template=%s | attn=%s",
                 self.name, self.load_s, self.has_chat_template, ATTN_IMPL)
        after = gpu_memory()
        if after:
            log.info("GPU after load: %s", after)
            free = min((v for k, v in after.items() if k.endswith("free_gib")), default=99.0)
            if free < 6.0:
                log.warning("only %.1f GiB free after loading weights; inference needs headroom. "
                            "Raise CFG.RESERVE_VRAM_GIB or lower CFG.MAX_VISUAL_TOKENS.", free)

    @classmethod
    def get(cls, cfg: Config = CFG) -> "QwenOCREngine":
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance


class MockEngine:
    """Rehearsal without a GPU. Returns the schema with every field null: it invents nothing."""
    name, has_chat_template, consecutive_ooms = "MOCK", True, 0


def free_model() -> None:
    """Release the model's VRAM. Call before re-loading in the same kernel."""
    inst = QwenOCREngine._instance
    if inst is not None:
        try:
            inst.model.to("meta")
        except Exception:
            pass
        del inst.model
        QwenOCREngine._instance = None
    release_cuda_cache()
    log.info("model freed. GPU: %s", gpu_memory() or "no CUDA")


def load_model(cfg: Config = CFG):
    """Load the model and processor EXACTLY ONCE. Never call this inside a loop."""
    return MockEngine() if cfg.MOCK_MODEL else QwenOCREngine.get(cfg)


ENGINE = load_model(CFG)
print("engine:", ENGINE.name)

In [ ]:
# =========================================================================
# CELL 6 — ZIP EXTRACTION
# =========================================================================
# Only the five required PDFs are written to disk. Everything seen is catalogued so the presence
# report can distinguish "absent" from "present under a name we failed to recognise".
def strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text))
                   if not unicodedata.combining(c))


def norm_name(text: str) -> str:
    """Accent-free, upper-case, punctuation-collapsed. Used ONLY for matching filenames."""
    s = re.sub(r"[^A-Z0-9]+", " ", strip_accents(text).upper())
    return re.sub(r"\s+", " ", s).strip()


# Real folders are messy: accents, underscores, trailing "(1)", and typos. The source list itself
# spells one file CARTON SIGNATUTE (a typo for SIGNATURE), so both spellings are accepted.
DOC_ALIASES: "OrderedDict[str, List[str]]" = OrderedDict([
    ("JUSTIFICATIF IDENTITE.PDF", ["JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE",
                                   "JUSTIFICATIF DE IDENTITE", "JUSTIF IDENTITE",
                                   "PIECE IDENTITE", "PIECE D IDENTITE", "IDENTITE"]),
    ("JUSTIFICATIF DOMICILE.PDF", ["JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE",
                                   "JUSTIF DOMICILE", "PREUVE DE DOMICILE", "DOMICILE"]),
    ("CONVENTION COMPTE.PDF", ["CONVENTION COMPTE", "CONVENTION DE COMPTE",
                               "CONVENTION DU COMPTE", "CONVENTION OUVERTURE COMPTE"]),
    ("FATCA.PDF", ["FATCA", "FORMULAIRE FATCA", "FATCA CRS", "AUTOCERTIFICATION FATCA"]),
    # SIGNATUTE is the spelling used in an earlier version of the spec; both are accepted so a
    # folder matches whichever way the file was actually named.
    ("CARTON SIGNATURE.PDF", ["CARTON SIGNATURE", "CARTON SIGNATUTE", "CARTON DE SIGNATURE",
                              "SPECIMEN SIGNATURE", "SPECIMEN DE SIGNATURE",
                              "SPICIMEN DE SIGNATURE"]),
])
DOC_NORM = {doc: sorted({norm_name(a) for a in [doc] + al}, key=len, reverse=True)
            for doc, al in DOC_ALIASES.items()}
DOC_SLUG = {doc: norm_name(doc).replace(" ", "_") for doc in DOC_ALIASES}


def match_document(filename: str, fuzzy_threshold: float = 0.88) -> Tuple[Optional[str], str, float]:
    """Return (canonical_document_name, match_type, score). Fuzzy hits are flagged for review."""
    p = Path(str(filename))
    if p.suffix.lower() != ".pdf":
        return None, "not_pdf", 0.0
    stem = re.sub(r"\s+\d+$", "", norm_name(p.stem)).strip()      # drop "(1)" style counters
    for doc, aliases in DOC_NORM.items():
        if stem in aliases:
            return doc, "exact", 1.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            if len(a) >= 5 and a in stem:
                return doc, "contains", 0.95
    best_doc, best = None, 0.0
    for doc, aliases in DOC_NORM.items():
        for a in aliases:
            r = difflib.SequenceMatcher(None, stem, a).ratio()
            if r > best:
                best_doc, best = doc, r
    return (best_doc, "fuzzy", round(best, 3)) if best >= fuzzy_threshold else (None, "no_match", 0.0)


def _safe_parts(member: str) -> Optional[List[str]]:
    """Reject absolute paths and .. traversal (zip-slip)."""
    name = member.replace("\\", "/")
    if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
        return None
    parts = [p for p in name.split("/") if p not in ("", ".")]
    return None if any(p == ".." for p in parts) else parts


def extract_zip(zip_path: Path = None, cfg: Config = CFG) -> pd.DataFrame:
    """Unzip, keeping ONLY the five required PDFs. Returns a catalogue of every member seen."""
    zip_path = Path(zip_path or cfg.zip_path)
    assert zip_path.exists(), f"ZIP not found: {zip_path}"
    rows, seen = [], {}
    t0 = time.perf_counter()

    with zipfile.ZipFile(zip_path) as zf:
        infos = [i for i in zf.infolist() if not i.is_dir()]
        parts_all = []
        for info in infos:
            parts = _safe_parts(info.filename)
            if parts is None:
                log.warning("unsafe zip member skipped: %s", info.filename)
            parts_all.append(parts)
        # strip a single wrapper folder if the whole archive sits under one
        roots = {p[0] for p in parts_all if p and len(p) > 1}
        root = roots.pop() if len(roots) == 1 and any(p and len(p) > 2 for p in parts_all) else None

        for info, parts in zip(infos, parts_all):
            if parts is None:
                continue
            if root and parts[0] == root:
                parts = parts[1:]
            if not parts:
                continue
            customer_id = parts[0] if len(parts) > 1 else "_ARCHIVE_ROOT_"
            fname = parts[-1]
            doc, match_type, score = match_document(fname)
            row = dict(customer_id=customer_id, member="/".join(parts), filename=fname,
                       matched_document=doc or "", match_type=match_type, match_score=score,
                       size_bytes=info.file_size, extracted_path="")
            if doc is not None:
                dest_dir = DIRS["pdfs"] / customer_id
                dest_dir.mkdir(parents=True, exist_ok=True)
                n = seen.get((customer_id, doc), 0)
                seen[(customer_id, doc)] = n + 1
                dest = dest_dir / (f"{DOC_SLUG[doc]}.pdf" if n == 0
                                   else f"{DOC_SLUG[doc]}__dup{n+1}.pdf")
                dest.write_bytes(zf.read(info))
                row["extracted_path"] = str(dest)
                row["is_duplicate"] = bool(n)
            rows.append(row)

    catalog = pd.DataFrame(rows)
    kept = int((catalog["matched_document"] != "").sum())
    log.info("zip: %d members, %d customer folders, %d required PDFs kept (%.1fs)",
             len(catalog), catalog["customer_id"].nunique(), kept, time.perf_counter() - t0)
    return catalog


# catalog_df = extract_zip()
print("CELL 6 ready: catalog_df = extract_zip()")

In [ ]:
# =========================================================================
# CELL 7 — CUSTOMER / DOCUMENT DISCOVERY
# =========================================================================
@dataclass
class CustomerDocs:
    customer_id: str
    documents: Dict[str, Optional[str]]        # canonical name -> extracted path or None
    n_files: int

    @property
    def has_identity(self) -> bool:
        return bool(self.documents.get(IDENTITY_DOC))

    @property
    def identity_pdf(self) -> Optional[Path]:
        p = self.documents.get(IDENTITY_DOC)
        return Path(p) if p else None


def discover_customers(catalog: pd.DataFrame) -> List[CustomerDocs]:
    """Group the catalogue by customer. One CustomerDocs per folder; no data crosses customers."""
    out = []
    for cid, grp in catalog.groupby("customer_id", sort=True):
        docs: Dict[str, Optional[str]] = {}
        for doc in DOCUMENTS_REQUIRED:
            hit = grp[(grp["matched_document"] == doc) & (grp["extracted_path"] != "")]
            docs[doc] = hit.iloc[0]["extracted_path"] if len(hit) else None
        out.append(CustomerDocs(customer_id=str(cid), documents=docs, n_files=len(grp)))
    return out


def select_identity_targets(customers: List[CustomerDocs],
                            cfg: Config = CFG) -> List[CustomerDocs]:
    """ONLY customers holding JUSTIFICATIF IDENTITE.PDF reach the model. The other four documents
    are reported present/absent and never sent to the GPU."""
    targets = [c for c in customers if c.has_identity]
    if cfg.LIMIT_CUSTOMERS:
        targets = targets[:cfg.LIMIT_CUSTOMERS]
    log.info("discovery: %d customers, %d with %s -> %d to process",
             len(customers), sum(c.has_identity for c in customers), IDENTITY_DOC, len(targets))
    return targets


print("CELL 7 ready: customers = discover_customers(catalog_df)")

In [ ]:
# =========================================================================
# CELL 8 — DOCUMENT PRESENCE REPORT
# =========================================================================
def build_presence_report(customers: List[CustomerDocs], catalog: pd.DataFrame,
                          cfg: Config = CFG) -> pd.DataFrame:
    """One row per customer, one column per required document (section 2)."""
    rows = []
    for c in customers:
        row: Dict[str, Any] = {"customer_id": c.customer_id}
        # Column names and TRUE/FALSE values exactly as specified in section 4.
        for doc, col in PRESENCE_COLUMNS.items():
            row[col] = "TRUE" if c.documents.get(doc) else "FALSE"
        grp = catalog[catalog["customer_id"] == c.customer_id]
        fuzzy = grp[grp["match_type"] == "fuzzy"]["filename"].tolist()
        row.update({
            "n_files_in_folder": c.n_files,
            "n_required_present": sum(bool(c.documents.get(d)) for d in DOCUMENTS_REQUIRED),
            "folder_complete": all(c.documents.get(d) for d in DOCUMENTS_REQUIRED),
            "will_be_processed": c.has_identity,
            "fuzzy_matched_filenames": ";".join(fuzzy),
            "needs_filename_review": bool(fuzzy),
        })
        rows.append(row)
    df = pd.DataFrame(rows)

    out_csv = DIRS["reports"] / "document_presence_report.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")     # BOM so Excel keeps accents/Arabic
    (DIRS["reports"] / "document_presence_report.json").write_text(
        df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
    catalog.to_csv(DIRS["reports"] / "archive_file_catalog.csv", index=False, encoding="utf-8-sig")

    log.info("presence report -> %s", out_csv)
    summary = pd.DataFrame({
        "present": [int((df[col] == "TRUE").sum()) for col in PRESENCE_COLUMNS.values()],
        "missing": [int((df[col] == "FALSE").sum()) for col in PRESENCE_COLUMNS.values()],
    }, index=list(PRESENCE_COLUMNS))
    summary["coverage_%"] = (100 * summary["present"] / max(1, len(df))).round(1)
    print(summary.to_string())
    return df


print("CELL 8 ready: presence_df = build_presence_report(customers, catalog_df)")

In [ ]:
# =========================================================================
# CELL 9 — PDF RENDERING UTILITIES
# =========================================================================
# EVERY page is rendered explicitly. The model never receives a PDF; it receives page images that
# this code produced, so "page 2 was processed" is verifiable rather than assumed.
@dataclass
class RenderedPage:
    page_number: int
    image: Image.Image
    width: int
    height: int
    render_dpi: int
    render_time_s: float
    pdf_rect: Tuple[float, float, float, float]    # page rect in PDF points, for clip re-render
    native_image_px: Optional[Tuple[int, int]] = None   # largest embedded raster, if any


def load_pdf(pdf_path: Path):
    """Open a PDF. Raises a clear error rather than returning a half-usable object."""
    if fitz is None:
        raise RuntimeError("PyMuPDF (pymupdf) is required for PDF rendering")
    doc = fitz.open(str(pdf_path))
    if doc.page_count == 0:
        doc.close()
        raise ValueError(f"PDF has zero pages: {pdf_path}")
    return doc


def _largest_embedded_image(doc, page_index: int) -> Optional[Tuple[int, int]]:
    """Native pixel size of the biggest raster on the page.

    A scanned PDF is usually one full-page JPEG. If that JPEG is 900 px wide, rendering at 600 dpi
    only interpolates -- knowing this stops us from spending tokens on an upsampled blur."""
    try:
        infos = doc[page_index].get_images(full=True)
        if not infos:
            return None
        return max(((i[2], i[3]) for i in infos), key=lambda wh: wh[0] * wh[1])
    except Exception:
        return None


def render_pdf_pages(pdf_path: Path, cfg: Config = CFG) -> List[RenderedPage]:
    """Render EVERY page independently at PDF_RENDER_DPI. No resizing happens here."""
    doc = load_pdf(pdf_path)
    pages: List[RenderedPage] = []
    try:
        n = min(doc.page_count, cfg.MAX_PAGES_PER_PDF)
        if doc.page_count > cfg.MAX_PAGES_PER_PDF:
            log.warning("%s has %d pages; capped at %d", pdf_path.name, doc.page_count, n)
        for i in range(n):
            t0 = time.perf_counter()
            page = doc[i]
            pix = page.get_pixmap(dpi=cfg.PDF_RENDER_DPI, colorspace=fitz.csRGB, alpha=False)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
            r = page.rect
            pages.append(RenderedPage(
                page_number=i + 1, image=img, width=pix.width, height=pix.height,
                render_dpi=cfg.PDF_RENDER_DPI,
                render_time_s=round(time.perf_counter() - t0, 3),
                pdf_rect=(r.x0, r.y0, r.x1, r.y1),
                native_image_px=_largest_embedded_image(doc, i)))
    finally:
        doc.close()
    log.info("%s: rendered %d pages (%s)", pdf_path.name, len(pages),
             ", ".join(f"p{p.page_number} {p.width}x{p.height} {p.render_time_s}s" for p in pages))
    return pages


def render_pdf_clip(pdf_path: Path, page_number: int, clip_rect: Tuple[float, float, float, float],
                    dpi: int) -> Optional[Image.Image]:
    """Re-render ONE RECTANGLE of a page at high DPI, straight from the PDF.

    This is the core of the MRZ fix. Upscaling a downscaled crop interpolates pixels that were
    already thrown away; re-rendering the clip recovers real detail from the source document, and
    a narrow strip costs a fraction of the visual tokens a full high-DPI page would."""
    if fitz is None:
        return None
    doc = None
    try:
        doc = fitz.open(str(pdf_path))
        page = doc[page_number - 1]
        rect = fitz.Rect(*clip_rect) & page.rect          # never ask for pixels outside the page
        if rect.is_empty or rect.width < 1 or rect.height < 1:
            return None
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=rect,
                              colorspace=fitz.csRGB, alpha=False)
        return Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
    except Exception as exc:
        log.warning("clip re-render failed p%d: %s", page_number, exc)
        return None
    finally:
        if doc is not None:
            doc.close()


print("CELL 9 ready: render_pdf_pages(), render_pdf_clip()")

In [ ]:
# =========================================================================
# CELL 10 — IMAGE PREPROCESSING AND ORIENTATION
# =========================================================================
# Nothing is applied unconditionally. Each operation has one measurement that triggers it, so a
# clean scan is left alone (CLAHE on a good scan thins strokes; a bilateral filter on a crisp
# 8.7 MP render costs seconds and gains nothing).
WORK_SIDE = 1000      # fixed measurement scale: blur variance is meaningless across sizes


def _work_gray(img: Image.Image, side: int = WORK_SIDE) -> np.ndarray:
    g = img.convert("L")
    if max(g.size) > side:
        s = side / max(g.size)
        g = g.resize((max(1, int(g.width * s)), max(1, int(g.height * s))), Image.BILINEAR)
    return np.asarray(g, dtype=np.uint8)


def estimate_noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer estimator: one 3x3 convolution, separates sensor noise from real structure."""
    if cv2 is None or gray.size == 0 or min(gray.shape) < 5:
        return 0.0
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray.astype(np.float32), -1, M)
    h, w = gray.shape
    return round(float(np.abs(conv).sum() * math.sqrt(0.5 * math.pi) /
                       (6.0 * (w - 2) * (h - 2))), 2)


def estimate_text_height_px(gray: np.ndarray) -> float:
    """Median height of text-like connected components. Predicts OCR failure far better than DPI:
    a 600 dpi scan of a tiny card and a 200 dpi scan of a form can carry the same glyph size."""
    if cv2 is None or gray.size == 0:
        return 0.0
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    n, _, stats, _ = cv2.connectedComponentsWithStats(binv, connectivity=8)
    H = gray.shape[0]
    hs = [stats[i][3] for i in range(1, n)
          if 3 <= stats[i][3] <= max(8, H * 0.08) and 1 <= stats[i][2] <= H * 0.15
          and stats[i][4] >= 6]
    return round(float(np.median(hs)), 1) if len(hs) >= 8 else 0.0


def _axis_score(binv: np.ndarray) -> float:
    """>1 = text lines run horizontally (0/180 deg); <1 = vertically (90/270)."""
    row, col = binv.sum(axis=1).astype(np.float32), binv.sum(axis=0).astype(np.float32)
    rv = row.var() / (row.mean() ** 2 + 1e-6)
    cvv = col.var() / (col.mean() ** 2 + 1e-6)
    return float((rv + 1e-6) / (cvv + 1e-6))


def _updown_score(binv: np.ndarray) -> float:
    """Ink-mass asymmetry inside text bands. Positive = upright, negative = upside down.

    Capitals and ascenders outnumber descenders in Latin and Cyrillic, and Arabic likewise carries
    most of its mass above the baseline, so the ink centroid of a text band sits ABOVE the band's
    geometric centre when the page is the right way up."""
    row = binv.sum(axis=1).astype(np.float32)
    if row.max() <= 0:
        return 0.0
    thr = row.mean() + 0.3 * row.std()
    bands, start = [], None
    for i, v in enumerate(row):
        if v > thr and start is None:
            start = i
        elif v <= thr and start is not None:
            if i - start >= 3:
                bands.append((start, i))
            start = None
    if start is not None and len(row) - start >= 3:
        bands.append((start, len(row)))
    if len(bands) < 3:
        return 0.0
    scores = []
    for a, b in bands[:60]:
        seg = row[a:b]
        if seg.sum() <= 0:
            continue
        idx = np.arange(len(seg), dtype=np.float32)
        centroid = float((seg * idx).sum() / seg.sum()) / max(1, len(seg) - 1)
        scores.append(0.5 - centroid)
    return float(np.mean(scores) * 2.0) if scores else 0.0


def detect_orientation(img: Image.Image, cfg: Config = CFG) -> Dict[str, Any]:
    """Rotation needed, in degrees counter-clockwise. Pure CV: ~20 ms, no GPU, no model call.

    A VLM call to ask 'which way up is this?' costs a full image prefill for four output tokens.
    Cascade: tesseract OSD if installed -> projection-profile axis test -> baseline asymmetry."""
    out = {"rotation": 0, "method": "disabled", "margin": 1.0, "osd_conf": None}
    if not cfg.ENABLE_ORIENTATION_CORRECTION or cv2 is None:
        return out
    if pytesseract is not None:
        try:
            small = img.copy()
            small.thumbnail((1000, 1000))
            osd = pytesseract.image_to_osd(small, output_type=TessOutput.DICT, config="--psm 0")
            conf = float(osd.get("orientation_conf", 0.0))
            out["osd_conf"] = conf
            if conf >= 2.0:
                return {**out, "rotation": int(osd.get("rotate", 0)) % 360,
                        "method": "tesseract_osd"}
        except Exception:
            pass
    gray = _work_gray(img, 800)
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    axis = _axis_score(binv)
    candidates = (0, 180) if axis >= 1.0 else (90, 270)
    scored = {d: _updown_score(np.ascontiguousarray(np.rot90(binv, k=d // 90) if d else binv))
              for d in candidates}
    best = max(scored, key=lambda d: scored[d])
    out.update({"rotation": int(best), "method": "cv_projection",
                "margin": round(abs(scored[candidates[0]] - scored[candidates[1]]), 4),
                "axis_score": round(float(axis), 3)})
    return out


def estimate_skew(img: Image.Image, cfg: Config = CFG) -> float:
    if cv2 is None or not cfg.ENABLE_DESKEW:
        return 0.0
    gray = _work_gray(img, 1000)
    binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    binv = cv2.dilate(binv, cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3)))
    coords = cv2.findNonZero(binv)
    if coords is None or len(coords) < 80:
        return 0.0
    ang = cv2.minAreaRect(coords)[-1]
    ang = ang + 90 if ang < -45 else (ang - 90 if ang > 45 else ang)
    return 0.0 if abs(ang) > cfg.DESKEW_MAX_DEG else round(float(ang), 2)


def analyze_page(rendered: "RenderedPage", cfg: Config = CFG) -> Dict[str, Any]:
    """STAGE 1: measure the page. No GPU, ~30 ms. Drives every downstream decision."""
    img = rendered.image
    gray = _work_gray(img)
    scale_back = max(img.size) / max(gray.shape) if max(gray.shape) else 1.0

    lap = float(cv2.Laplacian(gray, cv2.CV_64F).var()) if cv2 is not None else \
        float(np.diff(gray.astype(np.float32), axis=1).var())
    p5, p95 = np.percentile(gray, [5, 95])
    contrast = float((p95 - p5) / 255.0)
    brightness = float(gray.mean() / 255.0)
    noise = estimate_noise_sigma(gray)
    text_h = round(estimate_text_height_px(gray) * scale_back, 1)
    ink = 0.0
    if cv2 is not None:
        binv = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        ink = round(float((binv > 0).mean()), 4)
    g32 = gray.astype(np.float32)
    bh, bw = max(1, gray.shape[0] // 16), max(1, gray.shape[1] // 16)
    blocks = [g32[i:i + bh, j:j + bw].mean() for i in range(0, gray.shape[0] - bh + 1, bh)
              for j in range(0, gray.shape[1] - bw + 1, bw)]
    uniformity = round(float(np.std(blocks) / 255.0), 4) if blocks else 0.0

    blur_score = float(np.clip((lap - cfg.BLUR_VAR_POOR) /
                               max(1e-6, cfg.BLUR_VAR_GOOD - cfg.BLUR_VAR_POOR), 0, 1))
    contrast_score = float(np.clip(contrast / cfg.CONTRAST_LOW, 0, 1))
    noise_score = float(np.clip(1 - noise / max(1e-6, 2 * cfg.NOISE_SIGMA_HIGH), 0, 1))
    clarity = round(0.4 * blur_score + 0.3 * contrast_score + 0.3 * noise_score, 3)

    q = {
        "page_number": rendered.page_number,
        "width": rendered.width, "height": rendered.height,
        "render_dpi": rendered.render_dpi,
        "native_image_px": list(rendered.native_image_px) if rendered.native_image_px else None,
        "dpi_estimate": rendered.render_dpi,
        "text_height_px": text_h,
        "blur_laplacian_var": round(lap, 1), "blur_score": round(blur_score, 3),
        "contrast_raw": round(contrast, 3), "contrast_score": round(contrast_score, 3),
        "brightness_score": round(brightness, 3),
        "noise_sigma": noise, "noise_score": round(noise_score, 3),
        "illumination_uniformity": uniformity, "ink_ratio": ink,
        "visual_clarity": clarity,
    }
    # A page with a tiny ink ratio AND no text-like components carries nothing: skip the GPU.
    # Both conditions are required -- a small ID card on a white A4 has a genuinely tiny ink ratio.
    q["is_blank"] = bool(ink < cfg.INK_RATIO_BLANK and text_h <= 0)
    q["needs_contrast_enhancement"] = bool(cfg.ENABLE_CONTRAST_ENHANCEMENT and
                                           (contrast < cfg.CONTRAST_LOW or
                                            uniformity > cfg.ILLUMINATION_POOR))
    q["needs_denoise"] = bool(cfg.ENABLE_DENOISE and noise > cfg.NOISE_SIGMA_HIGH)
    q["needs_upscale"] = bool(0 < text_h < 11.0)
    q["quality"] = ("blank" if q["is_blank"] else "good" if clarity >= 0.75
                    else "fair" if clarity >= 0.45 else "poor")
    return q


def preprocess_page(img: Image.Image, quality: Dict[str, Any], cfg: Config = CFG,
                    max_dimension: Optional[int] = None) -> Tuple[Image.Image, Dict[str, Any]]:
    """Geometry first, then resize, then photometry ON THE SMALL IMAGE.

    Order matters for speed: the previous implementation ran CLAHE and a bilateral filter at full
    8.7 MP and only then downscaled, filtering pixels it was about to discard."""
    max_dimension = max_dimension or cfg.MAX_IMAGE_DIMENSION
    applied: List[str] = []
    meta: Dict[str, Any] = {"input_size": list(img.size), "rotation": 0, "skew_deg": 0.0}
    t0 = time.perf_counter()

    orient = detect_orientation(img, cfg)
    meta["orientation"] = orient
    if orient["rotation"]:
        img = img.rotate(orient["rotation"], expand=True)      # PIL rotates counter-clockwise
        meta["rotation"] = orient["rotation"]
        applied.append(f"rotate_{orient['rotation']}")

    skew = estimate_skew(img, cfg)
    if abs(skew) >= cfg.DESKEW_MIN_DEG and cv2 is not None:
        a = np.asarray(img)
        h, w = a.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), skew, 1.0)
        cos, sin = abs(M[0, 0]), abs(M[0, 1])
        nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
        M[0, 2] += nw / 2 - w / 2
        M[1, 2] += nh / 2 - h / 2
        img = Image.fromarray(cv2.warpAffine(a, M, (nw, nh), flags=cv2.INTER_CUBIC,
                                             borderMode=cv2.BORDER_REPLICATE))
        meta["skew_deg"] = skew
        applied.append(f"deskew_{skew:+.2f}")

    if max(img.size) > max_dimension:
        s = max_dimension / max(img.size)
        img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
        applied.append(f"resize_{max_dimension}")

    if cv2 is not None:
        if quality.get("needs_contrast_enhancement"):
            g = np.asarray(img.convert("L"))
            if quality.get("illumination_uniformity", 0) > cfg.ILLUMINATION_POOR:
                bg = cv2.GaussianBlur(g.astype(np.float32), (0, 0), sigmaX=max(g.shape) / 30.0)
                g = np.clip(g / (bg + 1e-3) * float(np.median(bg)), 0, 255).astype(np.uint8)
                applied.append("flatten_illumination")
            g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
            img = Image.fromarray(g).convert("RGB")
            applied.append("clahe")
        if quality.get("needs_denoise"):
            # Edge preserving ONLY. Median/Gaussian blur eats thin strokes and Arabic diacritics.
            img = Image.fromarray(cv2.bilateralFilter(np.asarray(img.convert("L")), 5, 45, 45)
                                  ).convert("RGB")
            applied.append("denoise")

    meta.update({"applied": applied, "output_size": list(img.size),
                 "approx_visual_tokens": int(img.width * img.height / 784),
                 "preprocess_time_s": round(time.perf_counter() - t0, 3)})
    return img, meta


print("CELL 10 ready: analyze_page(), preprocess_page(), detect_orientation()")

In [ ]:
# =========================================================================
# CELL 11 — MRZ DETECTION AND CROPPING  (the fix for the page-2 failure)
# =========================================================================
# No hard-coded coordinates. The band is LOCATED by its texture, which is the same for TD1, TD2
# and TD3 and survives different passport layouts: a dense, monospaced, full-width block of
# uniform-height glyphs. Detection runs AFTER orientation correction, so rotated pages work too.
#
# Classic morphological localisation:
#   blackhat  -> isolates dark text on a light background
#   Sobel-x   -> MRZ glyphs are vertical-stroke dense
#   closing   -> merges characters into continuous lines
#   closing   -> merges the 2-3 lines into one block
# Candidates are then scored on aspect ratio, width fraction, fill and line count.
@dataclass
class MRZCandidate:
    bbox: Tuple[int, int, int, int]      # x0, y0, x1, y1 in the PROCESSED page image
    score: float
    n_lines: int
    width_frac: float
    rel_y: float                         # vertical centre, 0 = top, 1 = bottom
    method: str


def detect_mrz(img: Image.Image, cfg: Config = CFG) -> Optional[MRZCandidate]:
    """Locate the MRZ band. Returns None when no plausible band is present (e.g. an ID card
    page with no MRZ) -- absence is a legitimate answer, not a failure."""
    if cv2 is None:
        return None
    W0, H0 = img.size
    work_w = 1200
    scale = min(1.0, work_w / W0)
    gray = np.asarray(img.convert("L").resize(
        (max(1, int(W0 * scale)), max(1, int(H0 * scale))), Image.BILINEAR), dtype=np.uint8)
    H, W = gray.shape

    gray_b = cv2.GaussianBlur(gray, (3, 3), 0)
    rect_k = cv2.getStructuringElement(cv2.MORPH_RECT, (max(9, W // 60), 5))
    sq_k = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

    blackhat = cv2.morphologyEx(gray_b, cv2.MORPH_BLACKHAT, rect_k)
    grad = np.absolute(cv2.Sobel(blackhat, cv2.CV_32F, 1, 0, ksize=-1))
    mn, mx = grad.min(), grad.max()
    grad = ((grad - mn) / (mx - mn + 1e-6) * 255).astype("uint8")
    grad = cv2.morphologyEx(grad, cv2.MORPH_CLOSE, rect_k)
    thresh = cv2.threshold(grad, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, sq_k)
    thresh = cv2.erode(thresh, None, iterations=2)
    # MRZ runs the full width; suppress the page borders so they cannot form a fake band
    thresh[:, :int(0.02 * W)] = 0
    thresh[:, int(0.98 * W):] = 0

    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best: Optional[MRZCandidate] = None
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if h < 4 or w < 0.45 * W:
            continue
        ar, wf = w / float(h), w / float(W)
        if ar < 4.0 or h > 0.30 * H:
            continue
        roi = thresh[y:y + h, x:x + w]
        fill = float((roi > 0).mean())
        if fill < 0.25:
            continue
        # count text lines inside the band: TD3/TD2 = 2, TD1 = 3
        prof = (roi > 0).sum(axis=1).astype(np.float32)
        lines, run = 0, False
        for v in prof:
            hot = v > 0.35 * w
            if hot and not run:
                lines, run = lines + 1, True
            elif not hot:
                run = False
        rel_y = (y + h / 2) / H
        score = (min(ar / 20.0, 1.0) * 0.30 + wf * 0.30 + fill * 0.15 +
                 (0.15 if lines in (2, 3) else 0.0) +
                 (0.10 if rel_y > 0.60 else 0.0))       # bottom is typical, not required
        cand = MRZCandidate(
            bbox=(int(x / scale), int(y / scale), int((x + w) / scale), int((y + h) / scale)),
            score=round(float(score), 3), n_lines=int(lines), width_frac=round(wf, 3),
            rel_y=round(float(rel_y), 3), method="morphological")
        if best is None or cand.score > best.score:
            best = cand
    if best is not None and best.score < 0.45:
        return None
    return best


def _pad_box(box: Tuple[int, int, int, int], W: int, H: int,
             pad: float) -> Tuple[int, int, int, int]:
    px, py = int(pad * W), int(pad * H)
    x0, y0, x1, y1 = box
    return (max(0, x0 - px), max(0, y0 - py), min(W, x1 + px), min(H, y1 + py))


def _box_to_pdf_rect(box: Tuple[int, int, int, int], proc_size: Tuple[int, int],
                     pdf_rect: Tuple[float, float, float, float],
                     rotation: int) -> Optional[Tuple[float, float, float, float]]:
    """Map a box in the PROCESSED image back to PDF point coordinates.

    Only exact for 0/90/180/270 rotations. After a deskew warp the mapping is no longer a simple
    rectangle, so the caller falls back to pixel upscaling rather than cropping the wrong region."""
    W, H = proc_size
    x0, y0, x1, y1 = box
    fx0, fy0, fx1, fy1 = x0 / W, y0 / H, x1 / W, y1 / H          # fractions of the rotated image
    r = rotation % 360
    if r == 0:
        a, b, c, d = fx0, fy0, fx1, fy1
    elif r == 90:      # image was rotated 90 deg CCW from the page
        a, b, c, d = fy0, 1 - fx1, fy1, 1 - fx0
    elif r == 180:
        a, b, c, d = 1 - fx1, 1 - fy1, 1 - fx0, 1 - fy0
    elif r == 270:
        a, b, c, d = 1 - fy1, fx0, 1 - fy0, fx1
    else:
        return None
    px0, py0, px1, py1 = pdf_rect
    pw, ph = px1 - px0, py1 - py0
    return (px0 + a * pw, py0 + b * ph, px0 + c * pw, py0 + d * ph)


def enhance_mrz_crop(img: Image.Image, cfg: Config = CFG,
                     threshold_variant: bool = False) -> Tuple[Image.Image, List[str]]:
    """Grayscale, contrast, mild sharpening. Tuned for OCR-B on a narrow strip.

    Sharpening here is a linear unsharp mask on an already high-resolution crop: it raises edge
    contrast, it does not manufacture glyph shapes. It is never used to make an unreadable
    character 'readable' -- the applied operations are recorded with the result so a reviewer can
    see exactly what the model was shown."""
    ops: List[str] = []
    if cv2 is None:
        return img, ops
    g = np.asarray(img.convert("L"))
    g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 4)).apply(g)
    ops.append("clahe")
    if cfg.ENABLE_SHARPENING:
        blur = cv2.GaussianBlur(g, (0, 0), 1.2)
        g = cv2.addWeighted(g, 1.5, blur, -0.5, 0)          # mild unsharp mask
        ops.append("unsharp")
    if threshold_variant:
        # Last resort only: binarisation destroys stroke detail and can close thin gaps.
        g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 31, 11)
        ops.append("adaptive_threshold")
    return Image.fromarray(g).convert("RGB"), ops


def crop_mrz(candidate: MRZCandidate, processed_img: Image.Image, rendered: "RenderedPage",
             pdf_path: Path, preprocess_meta: Dict[str, Any], cfg: Config = CFG,
             threshold_variant: bool = False) -> Optional[Dict[str, Any]]:
    """Produce the highest-resolution MRZ image obtainable, and say where it came from.

    Preference order:
      1. re-render the band from the PDF at MRZ_RENDER_DPI  (real resolution)
      2. upscale the rendered crop with Lanczos              (interpolated, flagged as such)
    """
    W, H = processed_img.size
    box = _pad_box(candidate.bbox, W, H, cfg.MRZ_BAND_PAD)
    source, img = None, None

    deskewed = abs(preprocess_meta.get("skew_deg", 0.0)) >= cfg.DESKEW_MIN_DEG
    if not deskewed:
        rect = _box_to_pdf_rect(box, (W, H), rendered.pdf_rect,
                                preprocess_meta.get("rotation", 0))
        if rect:
            clip = render_pdf_clip(pdf_path, rendered.page_number, rect, cfg.MRZ_RENDER_DPI)
            if clip is not None and clip.width >= 200:
                rot = preprocess_meta.get("rotation", 0) % 360
                if rot:
                    clip = clip.rotate(rot, expand=True)   # match the orientation we corrected to
                img, source = clip, "pdf_clip_rerender"

    if img is None:
        crop = processed_img.crop(box)
        f = max(1.0, min(cfg.MRZ_UPSCALE_FACTOR, cfg.MRZ_MAX_WIDTH / max(1, crop.width)))
        img = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
        source = "pixel_upscale"

    if img.width > cfg.MRZ_MAX_WIDTH:                      # a wider strip buys nothing, costs tokens
        s = cfg.MRZ_MAX_WIDTH / img.width
        img = img.resize((cfg.MRZ_MAX_WIDTH, max(1, int(img.height * s))), Image.LANCZOS)

    img, ops = enhance_mrz_crop(img, cfg, threshold_variant)
    # An MRZ is 2 lines (TD2/TD3) or 3 (TD1); never fewer. If the line detector undercounted,
    # assuming its number would overstate the character height and wrongly pass the resolution
    # check, so clamp to the physical minimum.
    n_lines = max(2, candidate.n_lines or 2)
    est_char_h = img.height / n_lines * 0.55          # glyph height vs line pitch
    return {
        "image": img, "source": source, "bbox": list(box), "ops": ops,
        "crop_width": img.width, "crop_height": img.height,
        "estimated_char_height_px": round(est_char_h, 1),
        "sufficient_resolution": bool(est_char_h >= cfg.MRZ_MIN_CHAR_HEIGHT_PX),
        "approx_visual_tokens": int(img.width * img.height / 784),
        "detector_score": candidate.score, "n_lines": candidate.n_lines,
        "n_lines_assumed": n_lines,
        "threshold_variant": threshold_variant,
    }


print("CELL 11 ready: detect_mrz(), crop_mrz(), enhance_mrz_crop()")

In [ ]:
# =========================================================================
# CELL 11b — GENERIC HIGH-RESOLUTION REGION EXTRACTION
# =========================================================================
# The same mechanism that rescues the MRZ also rescues handwriting and document titles:
# map the region back to PDF coordinates and RE-RENDER it from the vector source, instead of
# upscaling pixels that were already discarded by the full-page downscale.
def extract_region_high_res(box: Tuple[int, int, int, int], processed_img: Image.Image,
                            rendered: "RenderedPage", pdf_path: Path,
                            preprocess_meta: Dict[str, Any], dpi: int,
                            max_width: int, cfg: Config = CFG) -> Dict[str, Any]:
    """Return the highest-resolution image obtainable for a region, and say where it came from."""
    W, H = processed_img.size
    box = (max(0, box[0]), max(0, box[1]), min(W, box[2]), min(H, box[3]))
    img, source = None, None

    deskewed = abs(preprocess_meta.get("skew_deg", 0.0)) >= cfg.DESKEW_MIN_DEG
    if not deskewed:
        rect = _box_to_pdf_rect(box, (W, H), rendered.pdf_rect,
                                preprocess_meta.get("rotation", 0))
        if rect:
            clip = render_pdf_clip(pdf_path, rendered.page_number, rect, dpi)
            if clip is not None and clip.width >= 200:
                rot = preprocess_meta.get("rotation", 0) % 360
                if rot:
                    clip = clip.rotate(rot, expand=True)
                img, source = clip, "pdf_clip_rerender"
    if img is None:
        crop = processed_img.crop(box)
        f = max(1.0, min(3.0, max_width / max(1, crop.width)))
        img = crop.resize((int(crop.width * f), int(crop.height * f)), Image.LANCZOS)
        source = "pixel_upscale"
    if img.width > max_width:
        s = max_width / img.width
        img = img.resize((max_width, max(1, int(img.height * s))), Image.LANCZOS)
    return {"image": img, "source": source, "bbox": list(box),
            "dimensions": [img.width, img.height],
            "approx_visual_tokens": int(img.width * img.height / 784)}


def title_band(processed_img: Image.Image, cfg: Config = CFG) -> Tuple[int, int, int, int]:
    """Top band of the page, where document titles and headers live."""
    W, H = processed_img.size
    return (0, 0, W, int(H * cfg.TITLE_BAND_FRACTION))


def page_tiles(processed_img: Image.Image, n: int = 2) -> List[Tuple[int, int, int, int]]:
    """Split a page into horizontal tiles with a small overlap.

    Each tile is later rendered at the full MAX_IMAGE_DIMENSION, so a half-page tile carries
    roughly twice the linear resolution of the whole page. Layout-agnostic: no hard-coded field
    coordinates, which is what would break the moment a form template changes."""
    W, H = processed_img.size
    out, step = [], H / n
    for i in range(n):
        y0 = max(0, int(i * step - 0.05 * step))
        y1 = min(H, int((i + 1) * step + 0.05 * step))
        out.append((0, y0, W, y1))
    return out


print("CELL 11b ready: extract_region_high_res(), title_band(), page_tiles()")

In [ ]:
# =========================================================================
# CELL 12 — QWEN INFERENCE  (PRESERVED implementation, optimised)
# =========================================================================
# This is the SAME Transformers path established previously and verified against the local
# checkpoint -- it is optimised here, not replaced:
#   model class    : taken from config.json["architectures"], then AutoModelForImageTextToText,
#                    then AutoModelForVision2Seq  (CELL 5)
#   processor      : AutoProcessor with min_pixels/max_pixels  (CELL 5, one instance)
#   chat template  : processor.apply_chat_template(messages, add_generation_prompt=True) with
#                    content = [{"type": "image"}, {"type": "text", ...}]
#   image input    : processor(text=[...], images=[PIL.Image], return_tensors="pt")
#   generation     : model.generate(...) under torch.inference_mode()
#
# Generation settings and WHY (sections 27, 28):
#   do_sample=False .......... TEMPERATURE=0 expressed the way the library actually supports it.
#                              Sampling is the mechanism by which an unseen character gets chosen.
#   repetition_penalty=1.0 ... MUST stay 1.0. Above it, tokens already emitted are penalised,
#                              which corrupts exactly the data we care about: 1980, AA1122, and
#                              the <<<<< filler of an MRZ. Standard anti-degeneration advice is
#                              wrong for transcription.
#   no_repeat_ngram_size=0 ... same reason; it would forbid "<<" outright.
#   max_new_tokens ........... sized per schema (320 / 96 / 128), not a round number. The previous
#                              1024 was ~3x the useful ceiling.
#   brace-balance stop ....... ends the call the moment the JSON object closes.
#   "{" prefill .............. forces JSON immediately: no preamble, far fewer parse failures.
#   enable_thinking=False .... Qwen3 templates enable thinking by default; a <think> block burns
#                              400-900 tokens per page and gives the model room to guess.
@dataclass
class GenResult:
    text: str
    n_output_tokens: int
    inference_s: float
    token_texts: List[str] = field(default_factory=list)
    token_logprobs: List[float] = field(default_factory=list)
    truncated: bool = False
    error: Optional[str] = None
    error_kind: Optional[str] = None
    degraded: Optional[str] = None


def is_oom_error(exc: BaseException) -> bool:
    if torch is not None and isinstance(exc, getattr(torch.cuda, "OutOfMemoryError", ())):
        return True
    t = f"{type(exc).__name__}: {exc}".lower()
    return "out of memory" in t or "outofmemory" in t


class _BraceStop:
    def __init__(self, tokenizer, prompt_len: int, open_offset: int = 1):
        self.tok, self.prompt_len, self.open_offset, self.every = \
            tokenizer, prompt_len, open_offset, 8

    def __call__(self, input_ids, scores, **kw):
        n = input_ids.shape[0]
        done = torch.zeros(n, dtype=torch.bool, device=input_ids.device)
        gen_len = input_ids.shape[1] - self.prompt_len
        if gen_len < 8 or gen_len % self.every:
            return done
        for i in range(n):
            txt = self.tok.decode(input_ids[i, self.prompt_len:], skip_special_tokens=True)
            depth, in_str, esc = self.open_offset, False, False
            for c in txt:
                if in_str:
                    if esc:        esc = False
                    elif c == "\\": esc = True
                    elif c == '"': in_str = False
                    continue
                if c == '"':   in_str = True
                elif c == "{": depth += 1
                elif c == "}":
                    depth -= 1
                    if depth <= 0:
                        done[i] = True
                        break
        return done


class _LogprobRecorder:
    """Log-probability of each chosen token, as a LogitsProcessor.

    Not output_scores=True: that retains a [batch, vocab] tensor per step for the whole
    generation (~100 MB per sequence at a 150k vocab), held exactly when memory is tightest.
    Under greedy decoding the chosen token is the argmax, so one scalar per step is equivalent.
    This yields an OBJECTIVE confidence signal, unlike a self-reported one."""

    def __init__(self, batch_size: int):
        self.token_ids: List[List[int]] = [[] for _ in range(batch_size)]
        self.logprobs: List[List[float]] = [[] for _ in range(batch_size)]

    def __call__(self, input_ids, scores):
        lp = torch.log_softmax(scores.float(), dim=-1)
        top = lp.argmax(dim=-1)
        vals = lp.gather(1, top.unsqueeze(1)).squeeze(1)
        for i, (t, v) in enumerate(zip(top.tolist(), vals.tolist())):
            if i < len(self.token_ids):
                self.token_ids[i].append(int(t))
                self.logprobs[i].append(float(v))
        return scores


# Compact system instruction (~330 tokens, previously ~1200). It is re-prefilled on EVERY call,
# so each redundant sentence is paid for once per page (section 29).
OCR_SYSTEM = """You are a visual transcription engine. The supplied image is the only authoritative evidence.

RULES
- Transcribe only characters you can actually see. Never infer, reconstruct, normalize, translate, transliterate or complete unreadable text.
- Never use knowledge of names, country formats or check digits to fill a gap.
- Preserve the original script exactly. Preserve every < in an MRZ.
- When a field is requested in Latin letters, transcribe only Latin characters actually printed or written on the document. Never transliterate Arabic or Cyrillic into Latin.
- If any character of a value is illegible, the whole value is unreadable: return null.
- Handwriting is not assumed readable. Read it only if the strokes are visually distinguishable.
- confidence describes VISUAL CERTAINTY only, never plausibility:
  high = every character clearly legible; medium = slight ambiguity; low = significant ambiguity;
  unreadable = cannot be read, value must be null.
- Output JSON only. No markdown, no commentary, no reasoning."""


def _build_messages(system: str, user: str) -> List[Dict[str, Any]]:
    return [{"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user}]}]


def run_qwen_ocr(image: Image.Image, user_prompt: str, max_new_tokens: int,
                 engine=None, system: str = OCR_SYSTEM, cfg: Config = CFG,
                 _depth: int = 0) -> GenResult:
    """One inference call. The model and processor come from the CELL 5 singleton: nothing is
    re-initialised here, per customer, per PDF, per page or per retry."""
    engine = engine or ENGINE
    if isinstance(engine, MockEngine):
        body = {f: {"value": None, "confidence": "unreadable"} for f in IDENTITY_FIELDS}
        body["image_quality"] = {"overall": "unreadable", "reason": "MOCK_MODEL"}
        txt = json.dumps(body)
        return GenResult(text=txt, n_output_tokens=len(txt) // 4, inference_s=0.01)

    from transformers import StoppingCriteriaList, LogitsProcessorList
    messages = _build_messages(system, user_prompt)
    try:
        text = engine.processor.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True,
                                                    enable_thinking=not cfg.DISABLE_THINKING)
    except TypeError:                 # processors without the thinking switch
        text = engine.processor.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)
    text += "{"

    try:
        inputs = engine.processor(text=[text], images=[image], padding=True,
                                  return_tensors="pt").to(engine.model.device)
        prompt_len = inputs["input_ids"].shape[1]
        rec = _LogprobRecorder(1)
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = engine.model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, num_beams=1,
                repetition_penalty=cfg.REPETITION_PENALTY,
                no_repeat_ngram_size=cfg.NO_REPEAT_NGRAM_SIZE,
                stopping_criteria=StoppingCriteriaList(
                    [_BraceStop(engine.tokenizer, prompt_len, 1)]),
                logits_processor=LogitsProcessorList([rec]),
                return_dict_in_generate=True, output_scores=False,
                pad_token_id=getattr(engine.tokenizer, "pad_token_id", None)
                             or getattr(engine.tokenizer, "eos_token_id", None))
        dt = time.perf_counter() - t0
        ids = out.sequences[0, prompt_len:].detach().to("cpu")
        del out, inputs                # drop GPU refs; no empty_cache() per page (section 27)
        n = int(ids.shape[0])
        engine.consecutive_ooms = 0
        return GenResult(text="{" + engine.tokenizer.decode(ids, skip_special_tokens=True),
                         n_output_tokens=n, inference_s=round(dt, 3),
                         token_texts=[engine.tokenizer.decode([t]) for t in rec.token_ids[0][:n]],
                         token_logprobs=rec.logprobs[0][:n],
                         truncated=bool(n >= max_new_tokens))
    except Exception as exc:
        if not is_oom_error(exc):
            return GenResult(text="", n_output_tokens=0, inference_s=0.0,
                             error=f"{type(exc).__name__}: {exc}", error_kind="inference")
        release_cuda_cache()           # the one place empty_cache() is justified
        engine.consecutive_ooms += 1
        if _depth < cfg.OOM_MAX_DOWNSCALES:
            f = cfg.OOM_DOWNSCALE_FACTOR
            small = image.resize((max(64, int(image.width * f)), max(64, int(image.height * f))),
                                 Image.LANCZOS)
            log.warning("OOM; retrying at %d%% (%dx%d)", int(f * 100), small.width, small.height)
            r = run_qwen_ocr(small, user_prompt, max_new_tokens, engine, system, cfg, _depth + 1)
            r.degraded = f"oom_downscale_x{f ** (_depth + 1):.2f}"
            return r
        return GenResult(text="", n_output_tokens=0, inference_s=0.0,
                         error=f"{type(exc).__name__}: {exc}", error_kind="oom")


print("CELL 12 ready: run_qwen_ocr() -- preserved Transformers path")

In [ ]:
# =========================================================================
# CELL 12b — PROMPTS
# =========================================================================
def build_identity_prompt(fields: Optional[Sequence[str]] = None,
                          page_number: Optional[int] = None,
                          n_pages: Optional[int] = None) -> str:
    """Identity page extraction. Restricting `fields` on a retry shortens prompt AND output."""
    fields = list(fields or IDENTITY_FIELDS)
    where = f" (page {page_number} of {n_pages})" if page_number and n_pages else ""
    keys = ",\n".join(f'  "{f}": {{"value": null, "confidence": "unreadable"}}' for f in fields)
    latin = ('\n"surname_latin" and "given_names_latin" must contain ONLY Latin letters that are '
             'actually printed on the document. If the document shows an Arabic name, do NOT '
             'transliterate it.') if any(f.endswith("_latin") for f in fields) else ""
    return f"""Transcribe this identity document page{where}.
{latin}

Return exactly this JSON object, filling in only what is visible:
{{
{keys},
  "image_quality": {{"overall": "unreadable", "reason": null}}
}}

A field that is not on this page keeps value null. Return only the JSON object."""


MRZ_PROMPT = """This image is the machine readable zone (MRZ) of an identity document, cropped and enlarged.

Transcribe the MRZ lines exactly as printed. Preserve every < character. Do not insert spaces. Do not correct anything: check digits must NOT be used to decide an unclear character.
If any character is illegible, set value to null.

Return exactly:
{"mrz": {"value": null, "confidence": "unreadable"}, "image_quality": {"overall": "unreadable", "reason": null}}

Put all lines in one string separated by \\n. Return only the JSON object."""


# The model TRANSCRIBES the title; Python decides whether it matches. Asking a VLM "does this say
# X?" invites agreement -- a leading question is answered leadingly. Transcribe-then-compare keeps
# the judgement deterministic and auditable.
TITLE_PROMPT = """Transcribe the printed heading text at the top of this document image.

Return exactly:
{"title_text": {"value": null, "confidence": "unreadable"},
 "other_headings": {"value": null, "confidence": "unreadable"},
 "image_quality": {"overall": "unreadable", "reason": null}}

"title_text" is the largest or most prominent heading, transcribed exactly as written including any typography or spelling that looks unusual. "other_headings" may hold further heading lines separated by " | ". Do not summarise, translate or correct anything. Return only the JSON object."""


def build_name_prompt(handwritten_hint: bool = False, region_hint: str = "") -> str:
    """Latin surname / given names from a secondary document (sections 13, 14, 16, 18)."""
    hw = ("\nThe name may be HANDWRITTEN. Read the strokes directly. If a letter is ambiguous, "
          "the value is unreadable: return null. Never complete a name from context.\n"
          if handwritten_hint else "")
    where = f"\n{region_hint}" if region_hint else ""
    return f"""Read the client's name from this document image.{where}
{hw}
Return exactly:
{{"surname_latin": {{"value": null, "confidence": "unreadable", "text_type": null}},
 "given_names_latin": {{"value": null, "confidence": "unreadable", "text_type": null}},
 "image_quality": {{"overall": "unreadable", "reason": null}}}}

Transcribe ONLY Latin letters actually visible on this document. Do not transliterate Arabic or Cyrillic. Do not correct spelling. "text_type" is "printed", "handwritten" or "mixed".
Return only the JSON object."""


print("CELL 12b ready: prompts")

In [ ]:
# =========================================================================
# CELL 13 — JSON PARSING, SCHEMAS, NORMALISATION
# =========================================================================
# RAW vs NORMALIZED (section 33) is absolute:
#   raw_value        the transcription, never altered by anything downstream
#   normalized_value a comparison key only, produced here and used ONLY in stage B
# Nothing in stage B may write back into raw_value.
CONFIDENCES = ("high", "medium", "low", "unreadable")
CONF_RANK = {"high": 3, "medium": 2, "low": 1, "unreadable": 0}
NULLISH = {"", "null", "none", "n/a", "na", "unreadable", "not visible", "-", "--", "?"}


def empty_fields(fields: Sequence[str], reason: Optional[str] = None) -> Dict[str, Any]:
    d: Dict[str, Any] = {f: {"value": None, "confidence": "unreadable"} for f in fields}
    d["image_quality"] = {"overall": "unreadable", "reason": reason}
    return d


def _first_json_object(text: str) -> Optional[str]:
    if not text:
        return None
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    t = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", t.strip())
    start = t.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        c = t[i]
        if in_str:
            if esc:        esc = False
            elif c == "\\": esc = True
            elif c == '"': in_str = False
            continue
        if c == '"':   in_str = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return t[start:i + 1]
    return t[start:] + "}" * depth if depth > 0 else None


def _clean(v: Any, name: str = "") -> Optional[str]:
    if isinstance(v, (list, tuple)):
        v = ("\n" if name == "mrz" else " ").join(str(x) for x in v if x is not None)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = str(v)
    if not isinstance(v, str):
        return None
    s = v.strip()
    return None if s.lower() in NULLISH else s


def parse_json(raw_text: str, fields: Sequence[str],
               extra_keys: Sequence[str] = ()) -> Tuple[Dict[str, Any], bool, Optional[str]]:
    """Coerce model output onto the requested schema. SUBTRACTIVE ONLY: a missing field becomes
    null, an unknown confidence is downgraded (never promoted), a null value is forced to
    'unreadable'. Unparsable output fails CLOSED -- all null, never a partial guess."""
    fields = list(fields)
    block = _first_json_object(raw_text)
    if block is None:
        return empty_fields(fields, "no JSON in model output"), False, "no_json"
    try:
        obj = json.loads(block)
    except json.JSONDecodeError as exc:
        try:
            obj = json.loads(re.sub(r",\s*([}\]])", r"\1", block))
        except Exception:
            return empty_fields(fields, "invalid JSON"), False, f"json_error: {exc}"
    if not isinstance(obj, dict):
        return empty_fields(fields, "not an object"), False, "not_an_object"

    lower = {str(k).strip().lower(): k for k in obj}
    out: Dict[str, Any] = {}
    for f in list(fields) + list(extra_keys):
        key = lower.get(f) or (lower.get("machine_readable_zone") if f == "mrz" else None)
        node = obj.get(key) if key else None
        if isinstance(node, dict):
            value = _clean(node.get("value"), f)
            conf = str(node.get("confidence", "")).strip().lower()
            ttype = str(node.get("text_type", "") or "").strip().lower() or None
        else:
            value, conf, ttype = _clean(node, f), "", None
        if conf not in CONFIDENCES:
            conf = "low" if value is not None else "unreadable"    # downgrade, never promote
        if value is None:
            conf = "unreadable"
        entry: Dict[str, Any] = {"value": value, "confidence": conf}
        if ttype in ("printed", "handwritten", "mixed"):
            entry["text_type"] = ttype
        out[f] = entry

    iq = obj.get(lower.get("image_quality", ""), None)
    overall = str(iq.get("overall", "unreadable")).strip().lower() if isinstance(iq, dict) \
        else "unreadable"
    out["image_quality"] = {"overall": overall,
                            "reason": iq.get("reason") if isinstance(iq, dict) else None}
    return out, True, None


# ------------------------- normalisation (comparison only) -------------------------
def normalize_name(value: Optional[str]) -> Optional[str]:
    """Comparison key. NEVER stored in place of a raw value, never returned to the extraction."""
    if value is None:
        return None
    s = strip_accents(str(value)).upper()
    s = re.sub(r"[^A-Z ]+", " ", s)                 # drop digits and punctuation for comparison
    return re.sub(r"\s+", " ", s).strip() or None


def normalize_title(value: Optional[str]) -> str:
    if not value:
        return ""
    s = strip_accents(str(value)).upper()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]+", " ", s)).strip()


def field_record(raw: Dict[str, Any], page: Optional[int] = None,
                 source: Optional[str] = None, bbox: Optional[List[int]] = None,
                 gen: Optional[GenResult] = None, name: str = "") -> Dict[str, Any]:
    """Audit-trail record for one extracted field (section 36)."""
    value = raw.get("value")
    rec = {
        "raw_value": value,
        "normalized_value": normalize_name(value) if name.endswith("_latin") else (
            re.sub(r"\s+", " ", str(value)).strip() if value else None),
        "confidence": raw.get("confidence", "unreadable"),
        "text_type": raw.get("text_type"),
        "page_number": page,
        "source_region": source,
        "bbox": bbox,
        "match_status": None,          # filled in stage B; never alters raw_value
    }
    if gen is not None and value:
        tc = token_confidence(gen, name, value)
        if tc is not None:
            rec["token_confidence"] = tc
    return rec


def token_confidence(gen: GenResult, field_name: str, value: Optional[str]) -> Optional[float]:
    """Mean probability of the tokens that produced this value: an objective decoder-side signal,
    not a self-report. A model genuinely torn between 0 and O shows it here."""
    if not gen or not gen.token_texts or not value:
        return None
    anchor = gen.text.find(f'"{field_name}"')
    idx = gen.text.find(str(value), anchor if anchor >= 0 else 0)
    if idx < 0:
        return None
    start, end = idx, idx + len(str(value))
    offset = len(gen.text) - sum(len(t) for t in gen.token_texts)
    pos, probs = offset, []
    for tok, lp in zip(gen.token_texts, gen.token_logprobs):
        nxt = pos + len(tok)
        if nxt > start and pos < end:
            probs.append(math.exp(lp))
        pos = nxt
        if pos >= end:
            break
    return round(float(np.mean(probs)), 4) if probs else None


print("CELL 13 ready: parse_json(), normalize_name(), field_record()")

In [ ]:
# =========================================================================
# CELL 14 — VALIDATION  (detects errors; never rewrites a character)
# =========================================================================
DATE_RX = [
    (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{4})$"), ("d", "m", "y")),
    (re.compile(r"^(\d{4})[./\- ](\d{2})[./\- ](\d{2})$"), ("y", "m", "d")),
    (re.compile(r"^(\d{2})[./\- ](\d{2})[./\- ](\d{2})$"), ("d", "m", "yy")),
    (re.compile(r"^(\d{2})\s?([A-Z]{3})\s?(\d{4})$"), ("d", "mon", "y")),
]
MONTHS3 = {m: i + 1 for i, m in enumerate(["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL",
                                           "AUG", "SEP", "OCT", "NOV", "DEC"])}
HOMOGLYPHS = set("O0I1L5S8B2ZG6")
MRZ_LINE_SPECS = {"TD1": (3, 30), "TD2": (2, 36), "TD3": (2, 44)}


def parse_date(value: str) -> Optional[date]:
    """For consistency checks ONLY. The stored transcription is never replaced by this."""
    s = str(value).strip().upper()
    for rx, order in DATE_RX:
        m = rx.match(s)
        if not m:
            continue
        g = dict(zip(order, m.groups()))
        try:
            y = int(g["y"]) if "y" in g else (2000 + int(g["yy"]) if int(g["yy"]) < 30
                                              else 1900 + int(g["yy"]))
            mth = MONTHS3.get(g.get("mon"), 0) or int(g.get("m", 0))
            return date(y, mth, int(g["d"]))
        except Exception:
            return None
    return None


def mrz_check_digit(s: str) -> Optional[str]:
    w, total = (7, 3, 1), 0
    for i, c in enumerate(s):
        if c == "<":      v = 0
        elif c.isdigit(): v = int(c)
        elif c.isalpha(): v = ord(c.upper()) - 55
        else:             return None
        total += v * w[i % 3]
    return str(total % 10)


def validate_mrz(value: Optional[str]) -> Dict[str, Any]:
    """Structure, length, charset, check digits. REPORT ONLY (section 12).

    A failing check digit proves a character was misread; it does NOT reveal which character or
    what it should be. Using it to pick a replacement would be hallucination wearing a checksum."""
    r: Dict[str, Any] = {"status": "absent", "format": None, "line_lengths": [],
                         "checks": {}, "flags": []}
    if not value:
        return r
    lines = [l.strip().replace(" ", "") for l in str(value).splitlines() if l.strip()]
    r["line_lengths"] = [len(l) for l in lines]
    fmt = next((k for k, (n, ln) in MRZ_LINE_SPECS.items()
                if len(lines) == n and all(abs(len(l) - ln) <= 2 for l in lines)), None)
    r["format"] = fmt
    if fmt is None:
        r["status"] = "structure_invalid"
        r["flags"].append(f"unexpected_layout:{len(lines)}x{r['line_lengths']}")
        return r
    if any(re.search(r"[^A-Z0-9<]", l) for l in lines):
        r["flags"].append("unexpected_characters")
    try:
        if fmt in ("TD2", "TD3"):
            l2 = lines[1]
            for name, (a, b, cd) in {"document_number": (0, 9, 9), "date_of_birth": (13, 19, 19),
                                     "expiry_date": (21, 27, 27)}.items():
                if len(l2) > cd:
                    r["checks"][name] = {"computed": mrz_check_digit(l2[a:b]), "printed": l2[cd]}
        else:
            r["checks"]["document_number"] = {"computed": mrz_check_digit(lines[0][5:14]),
                                              "printed": lines[0][14]}
            r["checks"]["date_of_birth"] = {"computed": mrz_check_digit(lines[1][0:6]),
                                            "printed": lines[1][6]}
            r["checks"]["expiry_date"] = {"computed": mrz_check_digit(lines[1][8:14]),
                                          "printed": lines[1][14]}
    except Exception as exc:
        r["flags"].append(f"checksum_error:{exc}")
    verdicts = [c["computed"] == c["printed"] for c in r["checks"].values()
                if c.get("computed") is not None]
    r["status"] = ("not_verifiable" if not verdicts else
                   "valid" if all(verdicts) else "checksum_mismatch")
    if r["status"] == "checksum_mismatch":
        r["flags"].append("checksum_mismatch:" + ",".join(
            k for k, c in r["checks"].items() if c.get("computed") != c.get("printed")))
    r["note"] = "informational only; no character is corrected from a check digit"
    return r


def validate_identity_fields(fields: Dict[str, Any],
                             gen: Optional[GenResult] = None) -> Dict[str, Any]:
    """Format and cross-field checks over the raw transcription. Findings only."""
    v: Dict[str, Any] = {"fields": {}, "cross_field": {"checks": {}, "flags": []}, "mrz": {}}
    for name in IDENTITY_FIELDS:
        node = fields.get(name, {})
        val = node.get("raw_value", node.get("value"))
        e: Dict[str, Any] = {"format_valid": None, "flags": []}
        if val:
            if name in ("date_of_birth", "issue_date", "expiry_date"):
                d = parse_date(val)
                e["format_valid"] = d is not None
                if d is None:
                    e["flags"].append("unrecognised_date_format")
                else:
                    e["parsed_date"] = d.isoformat()
                    if name == "date_of_birth":
                        age = (date.today() - d).days / 365.25
                        if not 0 <= age <= 120:
                            e["format_valid"] = False
                            e["flags"].append(f"implausible_age:{age:.0f}")
                    if name == "expiry_date" and d < date.today():
                        e["flags"].append("document_expired")
            elif name in ("document_number", "personal_number"):
                core = str(val).upper().replace(" ", "")
                e["format_valid"] = bool(re.fullmatch(r"[A-Z0-9<\-/]{4,20}", core))
                risky = sorted(set(core) & HOMOGLYPHS)
                if risky and re.search(r"\d", core) and re.search(r"[A-Z]", core):
                    # FLAG ONLY. Replacing O with 0 here would be the pipeline guessing.
                    e["flags"].append("possible_ocr_confusion:" + "".join(risky))
                    e["needs_review"] = True
            elif name.endswith("_latin"):
                e["format_valid"] = bool(re.fullmatch(r"[A-Za-z '\-\.]+", str(val)))
                if not e["format_valid"]:
                    # A "latin" field holding non-Latin characters means the model ignored the
                    # instruction, or transliterated. Either way it is not trustworthy.
                    e["flags"].append("non_latin_characters_in_latin_field")
            elif name == "sex":
                e["format_valid"] = str(val).strip().upper() in {
                    "M", "F", "X", "H", "MALE", "FEMALE", "MASCULIN", "FEMININ"}
        v["fields"][name] = e

    v["mrz"] = validate_mrz((fields.get("mrz") or {}).get("raw_value"))
    dob = parse_date((fields.get("date_of_birth") or {}).get("raw_value") or "")
    iss = parse_date((fields.get("issue_date") or {}).get("raw_value") or "")
    exp = parse_date((fields.get("expiry_date") or {}).get("raw_value") or "")
    cf = v["cross_field"]
    if dob and iss and dob >= iss:
        cf["flags"].append("birth_after_issue")
    if iss and exp and iss >= exp:
        cf["flags"].append("issue_after_expiry")
    cf["checks"] = {"birth_before_issue": (dob < iss) if (dob and iss) else None,
                    "issue_before_expiry": (iss < exp) if (iss and exp) else None}

    mrz_val = (fields.get("mrz") or {}).get("raw_value")
    if mrz_val:
        flat = re.sub(r"[^A-Z0-9]", "", str(mrz_val).upper())
        for f in ("document_number", "surname_latin"):
            fv = (fields.get(f) or {}).get("raw_value")
            if fv:
                key = re.sub(r"[^A-Z0-9]", "", strip_accents(str(fv)).upper())
                ok = key in flat
                cf["checks"][f"mrz_agrees_with_{f}"] = ok
                if not ok:
                    cf["flags"].append(f"mrz_disagrees_with_{f}")
    return v


print("CELL 14 ready: validate_mrz(), validate_identity_fields()")

In [ ]:
# =========================================================================
# CELL 15 — DOCUMENT CLASSIFICATION / VISUAL VERIFICATION
# =========================================================================
# A filename is not evidence (section 19). Each document's identity is verified from its printed
# title. The model TRANSCRIBES the heading; Python decides MATCH / MISMATCH / UNCERTAIN.
#
# Why not ask the model "is this the signature card?": a leading question gets a leading answer.
# Transcribe-then-compare keeps the decision deterministic, auditable and reproducible.
#
# Note on section 17: the expected title is given as "SPICIMEN DE SIGNATURE", which appears to be
# a typo for "SPECIMEN". Both spellings are accepted, and the title actually seen is recorded
# verbatim so the discrepancy stays visible rather than being silently normalised away.
EXPECTED_TITLES: Dict[str, Dict[str, Any]] = {
    IDENTITY_DOC: {
        "any_of": ["PASSEPORT", "PASSPORT", "CARTE NATIONALE D IDENTITE", "CARTE D IDENTITE",
                   "TITRE DE SEJOUR", "CARTE DE SEJOUR", "PERMIS DE CONDUIRE",
                   "REPUBLIQUE", "IDENTITY CARD"],
        "label": "identity document"},
    DOMICILE_DOC: {
        "any_of": ["FACTURE", "QUITTANCE", "ATTESTATION DE DOMICILE", "JUSTIFICATIF DE DOMICILE",
                   "ELECTRICITE", "RELEVE", "CONTRAT DE BAIL", "AVIS D ECHEANCE", "INVOICE"],
        "label": "proof of address"},
    CONVENTION_DOC: {
        "any_of": ["CONVENTION DE COMPTE", "CONVENTION COMPTE", "OUVERTURE DE COMPTE",
                   "CONDITIONS GENERALES", "CONVENTION DE GESTION"],
        "label": "account convention"},
    SIGNATURE_DOC: {
        "any_of": ["SPICIMEN DE SIGNATURE", "SPECIMEN DE SIGNATURE", "SPECIMEN SIGNATURE"],
        "label": "signature card"},
}

# FATCA has TWO components with different expected titles (section 15).
FATCA_COMPONENTS = {
    # The contiguous phrase matters: "FATCA" plus "IDENTIFICATION" as loose tokens also appears
    # in component 2's title ("FORMULAIRE D'IDENTIFICATION ... LOI FATCA"), which would make
    # component 1 swallow component 2's page.
    "component_1": {"all_of": ["FATCA IDENTIFICATION FORMS"],
                    "any_of": ["FATCA IDENTIFICATION FORMS"],
                    "expected_pages": 2,
                    "label": "FATCA Identification forms(v1.0)"},
    # "<>" in the specified title is treated as a wildcard: the two anchor phrases must both be
    # present, whatever sits between them.
    "component_2": {"all_of": ["FORMULAIRE D IDENTIFICATION", "AU REGARD DE LA LOI FATCA"],
                    "any_of": ["SOUSCRIPTEUR", "ASSURE", "AU REGARD DE LA LOI FATCA"],
                    "expected_pages": 1,
                    "label": "FORMULAIRE D'IDENTIFICATION <> AU REGARD DE LA LOI FATCA "
                             "SOUSCRIPTEUR/ASSURE"},
}


def read_page_title(rendered: "RenderedPage", processed: Image.Image, pdf_path: Path,
                    pre_meta: Dict[str, Any], engine, timings: Dict[str, float],
                    cfg: Config = CFG) -> Dict[str, Any]:
    """Transcribe the heading band of one page, re-rendered at TITLE_RENDER_DPI."""
    t0 = time.perf_counter()
    region = extract_region_high_res(title_band(processed, cfg), processed, rendered, pdf_path,
                                     pre_meta, cfg.TITLE_RENDER_DPI,
                                     cfg.MAX_IMAGE_DIMENSION, cfg)
    timings["preprocessing"] += time.perf_counter() - t0
    gen = run_qwen_ocr(region["image"], TITLE_PROMPT, 128, engine, cfg=cfg)
    timings["title_inference"] += gen.inference_s
    parsed, ok, _ = parse_json(gen.text, ["title_text", "other_headings"])
    return {
        "title_text": parsed["title_text"]["value"],
        "title_confidence": parsed["title_text"]["confidence"],
        "other_headings": parsed["other_headings"]["value"],
        "region_source": region["source"], "region_dimensions": region["dimensions"],
        "inference_s": gen.inference_s, "parse_ok": ok, "error_kind": gen.error_kind,
        "raw_output": gen.text[:1000],
    }


def classify_title(observed: Optional[str], other: Optional[str],
                   spec: Dict[str, Any], cfg: Config = CFG) -> Dict[str, Any]:
    """Deterministic comparison of an observed title against an expectation.

    UNCERTAIN when nothing could be read (an unreadable title is not a wrong document);
    MISMATCH only when a title WAS read and it does not correspond."""
    haystack = normalize_title(" | ".join([x for x in (observed, other) if x]))
    if not haystack:
        return {"status": "UNCERTAIN", "observed_title": observed, "score": None,
                "reason": "title not readable"}

    hits, scores = [], []
    for phrase in spec.get("any_of", []):
        n = normalize_title(phrase)
        if n and n in haystack:
            hits.append(phrase)
            scores.append(1.0)
        else:
            best = max((difflib.SequenceMatcher(None, n, haystack[i:i + len(n)]).ratio()
                        for i in range(0, max(1, len(haystack) - len(n) + 1), 4)), default=0.0)
            scores.append(best)
    required = [p for p in spec.get("all_of", []) if normalize_title(p) not in haystack]
    score = round(max(scores) if scores else 0.0, 3)

    if spec.get("all_of") and required:
        status = "MISMATCH"
        reason = "missing required phrase: " + "; ".join(required)
    elif hits or score >= cfg.TITLE_MATCH_THRESHOLD:
        status = "MATCH"
        reason = ("exact phrase found: " + hits[0]) if hits else f"fuzzy score {score}"
    else:
        status = "MISMATCH"
        reason = f"no expected phrase found (best score {score} < {cfg.TITLE_MATCH_THRESHOLD})"
    return {"status": status, "observed_title": observed, "matched_phrase": hits[0] if hits else None,
            "score": score, "threshold": cfg.TITLE_MATCH_THRESHOLD, "reason": reason}


print("CELL 15 ready: read_page_title(), classify_title()")

In [ ]:
# =========================================================================
# CELL 16 — STAGE A: IDENTITY DOCUMENT  (multi-page + dedicated MRZ path)
# =========================================================================
def _needs_retry(fields: Dict[str, Any], names: Sequence[str], minimum: str = "medium") -> List[str]:
    lim = CONF_RANK[minimum]
    return [f for f in names
            if CONF_RANK.get(fields.get(f, {}).get("confidence", "unreadable"), 0) < lim]


def process_identity_page(rendered: RenderedPage, pdf_path: Path, engine,
                          timings: Dict[str, float], errors: List[Dict[str, Any]],
                          customer_id: str, n_pages: int,
                          cfg: Config = CFG) -> Dict[str, Any]:
    """One identity page: analysis -> full-page extraction -> targeted field retry.

    EVERY page runs through this, including page 2 and beyond. There is no assumption that the
    interesting content is on page 1."""
    t_page = time.perf_counter()
    rec: Dict[str, Any] = {"customer_id": customer_id, "document": IDENTITY_DOC,
                           "page_number": rendered.page_number,
                           "render_time": rendered.render_time_s,
                           "image_width": rendered.width, "image_height": rendered.height,
                           "native_image_px": list(rendered.native_image_px)
                           if rendered.native_image_px else None}

    t0 = time.perf_counter()
    quality = analyze_page(rendered, cfg)
    processed, pre_meta = preprocess_page(rendered.image, quality, cfg)
    mrz_cand = None if quality["is_blank"] else detect_mrz(processed, cfg)
    prep_s = time.perf_counter() - t0
    timings["preprocessing"] += prep_s
    rec.update({"preprocessing_time": round(prep_s, 3), "page_quality": quality,
                "preprocessing": pre_meta, "processed_dimensions": list(processed.size),
                "mrz_detected": bool(mrz_cand),
                "mrz_detection": asdict(mrz_cand) if mrz_cand else None})
    rec["_processed_image"] = processed

    if quality["is_blank"]:
        rec.pop("_processed_image", None)
        rec.update({"status": "SKIPPED_BLANK", "inference_time": 0.0, "n_model_calls": 0,
                    "fields": empty_fields(IDENTITY_FIELDS, "blank page"),
                    "processing_time": round(time.perf_counter() - t_page, 2)})
        return rec

    # ---- PASS 1: fast general extraction ----
    gen = run_qwen_ocr(processed, build_identity_prompt(None, rendered.page_number, n_pages),
                       cfg.MAX_NEW_TOKENS, engine, cfg=cfg)
    timings["inference"] += gen.inference_s
    calls, infer = 1, gen.inference_s
    t0 = time.perf_counter()
    fields, ok, warn = parse_json(gen.text, IDENTITY_FIELDS)
    timings["json_parse"] += time.perf_counter() - t0
    status = "OK" if ok else "PARSE_FAILED"
    if gen.error_kind:
        status = "SYSTEM_ERROR"
        errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                       "file": pdf_path.name, "page": rendered.page_number,
                       "stage": "identity_pass1", "error": gen.error, "timestamp": utcnow()})

    # ---- PASS 2: targeted retry of the failed fields ONLY, at higher effective resolution ----
    retried: List[str] = []
    if status == "OK" and cfg.MAX_FIELD_RETRIES > 0:
        missing = [f for f in _needs_retry(fields, ["surname_latin", "given_names_latin",
                                                    "date_of_birth", "document_number"])]
        if missing and quality["visual_clarity"] > 0.12:
            # Tile the page: a half page rendered at the same max dimension carries roughly twice
            # the linear resolution, which is exactly what small print needs.
            for box in page_tiles(processed, cfg.HANDWRITING_TILES)[:cfg.MAX_FIELD_RETRIES + 1]:
                region = extract_region_high_res(box, processed, rendered, pdf_path, pre_meta,
                                                 cfg.HANDWRITING_RENDER_DPI,
                                                 cfg.MAX_IMAGE_DIMENSION, cfg)
                r2 = run_qwen_ocr(region["image"], build_identity_prompt(missing), 
                                  cfg.MAX_NEW_TOKENS_TARGETED, engine, cfg=cfg)
                timings["inference"] += r2.inference_s
                calls += 1
                infer += r2.inference_s
                f2, ok2, _ = parse_json(r2.text, missing)
                if ok2:
                    for f in list(missing):
                        if CONF_RANK.get(f2[f]["confidence"], 0) > \
                           CONF_RANK.get(fields[f]["confidence"], 0):
                            fields[f] = f2[f]
                            retried.append(f)
                missing = _needs_retry(fields, missing)
                if not missing:
                    break

    rec.pop("_processed_image", None)
    rec["_processed_image"] = processed
    rec.update({"status": status, "fields": fields, "raw_output": gen.text[:4000],
                "parse_warning": warn, "retried_fields": sorted(set(retried)),
                "inference_time": round(infer, 3), "n_model_calls": calls,
                "n_output_tokens": gen.n_output_tokens, "degraded": gen.degraded,
                "error_kind": gen.error_kind, "_gen": gen,
                "processing_time": round(time.perf_counter() - t_page, 2)})
    return rec


def process_mrz(page_rec: Dict[str, Any], rendered: RenderedPage, processed: Image.Image,
                pdf_path: Path, engine, timings: Dict[str, float],
                errors: List[Dict[str, Any]], customer_id: str,
                cfg: Config = CFG) -> Optional[Dict[str, Any]]:
    """DEDICATED MRZ PATH (section 11). Only the MRZ is retried -- never the whole document."""
    cand_d = page_rec.get("mrz_detection")
    if not cand_d:
        return None
    cand = MRZCandidate(bbox=tuple(cand_d["bbox"]), score=cand_d["score"],
                        n_lines=cand_d["n_lines"], width_frac=cand_d["width_frac"],
                        rel_y=cand_d["rel_y"], method=cand_d["method"])
    report: Dict[str, Any] = {
        "mrz_detected": True, "mrz_page": rendered.page_number,
        "mrz_detector_score": cand.score, "mrz_crop_width": None, "mrz_crop_height": None,
        "mrz_inference_time": 0.0, "mrz_validation_status": None, "mrz_raw_output": None,
        "mrz_value": None, "mrz_confidence": "unreadable", "mrz_source": None, "attempts": []}

    for attempt in range(1, cfg.MAX_MRZ_ATTEMPTS + 1):
        t0 = time.perf_counter()
        crop = crop_mrz(cand, processed, rendered, pdf_path, page_rec["preprocessing"], cfg,
                        threshold_variant=(attempt == 3 and cfg.ENABLE_MRZ_THRESHOLD_VARIANT))
        if attempt == 2 and crop and crop["source"] == "pdf_clip_rerender":
            box = _pad_box(cand.bbox, *processed.size, cfg.MRZ_BAND_PAD)
            c = processed.crop(box)
            f = max(1.0, min(cfg.MRZ_UPSCALE_FACTOR, cfg.MRZ_MAX_WIDTH / max(1, c.width)))
            up, ops = enhance_mrz_crop(
                c.resize((int(c.width * f), int(c.height * f)), Image.LANCZOS), cfg)
            crop = {**crop, "image": up, "source": "pixel_upscale", "ops": ops,
                    "crop_width": up.width, "crop_height": up.height}
        timings["preprocessing"] += time.perf_counter() - t0
        if crop is None:
            report["mrz_detected"] = False
            break

        gen = run_qwen_ocr(crop["image"], MRZ_PROMPT, cfg.MAX_NEW_TOKENS_MRZ, engine, cfg=cfg)
        timings["mrz_inference"] += gen.inference_s
        report["mrz_inference_time"] = round(report["mrz_inference_time"] + gen.inference_s, 3)
        parsed, ok, _ = parse_json(gen.text, ["mrz"])
        value = parsed["mrz"]["value"]
        validation = validate_mrz(value)
        report["attempts"].append({
            "attempt": attempt, "source": crop["source"], "ops": crop["ops"],
            "crop": [crop["crop_width"], crop["crop_height"]],
            "estimated_char_height_px": crop["estimated_char_height_px"],
            "sufficient_resolution": crop["sufficient_resolution"],
            "visual_tokens": crop["approx_visual_tokens"],
            "inference_s": round(gen.inference_s, 3), "got_value": value is not None,
            "validation_status": validation["status"]})
        report.update({"mrz_crop_width": crop["crop_width"], "mrz_crop_height": crop["crop_height"],
                       "mrz_raw_output": gen.text[:2000], "mrz_source": crop["source"],
                       "mrz_validation_status": validation["status"],
                       "mrz_validation": validation})
        if gen.error_kind:
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "file": pdf_path.name, "page": rendered.page_number,
                           "stage": f"mrz_attempt_{attempt}", "error": gen.error,
                           "timestamp": utcnow()})
            break
        if value is not None:
            report.update({"mrz_value": value, "mrz_confidence": parsed["mrz"]["confidence"]})
            # A checksum mismatch does NOT trigger another attempt. The transcription stands as
            # read and is flagged; retrying until the checksum passes would be guessing.
            break
        if cfg.SAVE_DEBUG_ON_FAILURE and attempt == cfg.MAX_MRZ_ATTEMPTS:
            d = DIRS["debug"] / customer_id
            d.mkdir(parents=True, exist_ok=True)
            crop["image"].save(d / f"p{rendered.page_number}_mrz_a{attempt}.png")
    return report


def process_identity_document(customer_id: str, pdf_path: Path, engine,
                              cfg: Config = CFG) -> Dict[str, Any]:
    """Render and process EVERY page, then aggregate. The MRZ usually lives on page 2."""
    t_doc = time.perf_counter()
    timings: Dict[str, float] = defaultdict(float)
    errors: List[Dict[str, Any]] = []
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] = time.perf_counter() - t0

    page_records, mrz_reports, title_checks = [], [], []
    for rendered in pages:
        try:
            rec = process_identity_page(rendered, pdf_path, engine, timings, errors,
                                        customer_id, len(pages), cfg)
            processed = rec.pop("_processed_image", None)
            gen = rec.pop("_gen", None)
            if rec.get("mrz_detected") and processed is not None and rec["status"] != "SKIPPED_BLANK":
                mrz = process_mrz(rec, rendered, processed, pdf_path, engine, timings,
                                  errors, customer_id, cfg)
                rec["mrz_report"] = mrz
                if mrz:
                    mrz_reports.append(mrz)
            if cfg.VERIFY_DOCUMENT_TITLES and rendered.page_number == 1 and processed is not None:
                title_checks.append(read_page_title(rendered, processed, pdf_path,
                                                    rec["preprocessing"], engine, timings, cfg))
            rec["_gen_ref"] = gen
            page_records.append(rec)
        except Exception as exc:
            errors.append({"customer_id": customer_id, "document": IDENTITY_DOC,
                           "file": pdf_path.name, "page": rendered.page_number,
                           "stage": "process_identity_page",
                           "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})
            log.error("identity page %d failed for %s: %s", rendered.page_number, customer_id, exc)
            page_records.append({"page_number": rendered.page_number, "status": "PAGE_ERROR",
                                 "fields": empty_fields(IDENTITY_FIELDS, str(exc)),
                                 "inference_time": 0.0, "n_model_calls": 0})

    # ---- aggregate across pages: best confidence wins; equal-confidence disagreement -> null ----
    final: Dict[str, Any] = {}
    conflicts: List[Dict[str, Any]] = []
    for f in IDENTITY_FIELDS:
        cands = [{"page": p.get("page_number"), **p["fields"][f]} for p in page_records
                 if p.get("fields", {}).get(f, {}).get("value") is not None]
        if not cands:
            final[f] = field_record({"value": None, "confidence": "unreadable"}, name=f)
            continue
        cands.sort(key=lambda c: -CONF_RANK.get(c["confidence"], 0))
        best = cands[0]
        rivals = [c for c in cands[1:]
                  if normalize_name(c["value"]) != normalize_name(best["value"])]
        if [c for c in rivals if CONF_RANK.get(c["confidence"], 0) >=
                CONF_RANK.get(best["confidence"], 0)]:
            conflicts.append({"field": f, "candidates": cands})
            final[f] = field_record({"value": None, "confidence": "unreadable"}, name=f)
            final[f]["conflict"] = True
            continue
        gen_ref = next((p.get("_gen_ref") for p in page_records
                        if p.get("page_number") == best["page"]), None)
        final[f] = field_record(best, page=best["page"], source="full_page", gen=gen_ref, name=f)

    for r in mrz_reports:                # the dedicated pass saw 2-3x the character resolution
        if r and r.get("mrz_value"):
            final["mrz"] = field_record({"value": r["mrz_value"], "confidence": r["mrz_confidence"]},
                                        page=r["mrz_page"], source=r["mrz_source"], name="mrz")
            break

    for p in page_records:
        p.pop("_gen_ref", None)
    validation = validate_identity_fields(final)
    title = classify_title(title_checks[0]["title_text"] if title_checks else None,
                           title_checks[0].get("other_headings") if title_checks else None,
                           EXPECTED_TITLES[IDENTITY_DOC], cfg) if title_checks else \
        {"status": "UNCERTAIN", "reason": "title check disabled"}

    return {
        "document": IDENTITY_DOC, "source_pdf": str(pdf_path), "page_count": len(pages),
        "expected_document": EXPECTED_TITLES[IDENTITY_DOC]["label"],
        "detected_document": title.get("observed_title"),
        "document_verification_status": title["status"], "title_check": title,
        "fields": final, "validation": validation, "conflicts": conflicts,
        "mrz": mrz_reports[0] if mrz_reports else {"mrz_detected": False},
        "pages": page_records, "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
        "n_model_calls": sum(p.get("n_model_calls", 0) for p in page_records)
                         + sum(len(r.get("attempts", [])) for r in mrz_reports)
                         + len(title_checks),
        "processing_time_seconds": round(time.perf_counter() - t_doc, 2),
        "errors": errors,
    }


print("CELL 16 ready: process_identity_document()")

In [ ]:
# =========================================================================
# CELL 17 — STAGE A: SECONDARY DOCUMENTS (domicile, convention, FATCA, signature card)
# =========================================================================
# Same principle everywhere: verify the document visually, then read the Latin name FROM THIS
# DOCUMENT. A name is never copied from the identity document or the database to fill a gap here
# -- that would turn a verification signal into a self-fulfilling one.
def extract_names_from_page(rendered: RenderedPage, processed: Image.Image, pre_meta: Dict[str, Any],
                            pdf_path: Path, engine, timings: Dict[str, float],
                            handwritten: bool, cfg: Config = CFG) -> Tuple[Dict[str, Any], int, float]:
    """PASS 1 full page, then PASS 2 on high-resolution tiles for the fields still missing."""
    prompt = build_name_prompt(handwritten)
    gen = run_qwen_ocr(processed, prompt, cfg.MAX_NEW_TOKENS_TARGETED, engine, cfg=cfg)
    timings["inference"] += gen.inference_s
    calls, infer = 1, gen.inference_s
    fields, ok, _ = parse_json(gen.text, NAME_FIELDS)
    sources = {f: "full_page" for f in NAME_FIELDS}
    bboxes: Dict[str, Optional[List[int]]] = {f: None for f in NAME_FIELDS}
    gens = {f: gen for f in NAME_FIELDS}

    missing = _needs_retry(fields, NAME_FIELDS)
    if missing and ok:
        # Handwriting needs stroke detail, so the tile is RE-RENDERED from the PDF rather than
        # upscaled: a half page at the same max dimension carries ~2x the linear resolution.
        for box in page_tiles(processed, cfg.HANDWRITING_TILES)[:cfg.MAX_HANDWRITING_ATTEMPTS]:
            t0 = time.perf_counter()
            region = extract_region_high_res(box, processed, rendered, pdf_path, pre_meta,
                                             cfg.HANDWRITING_RENDER_DPI,
                                             cfg.MAX_IMAGE_DIMENSION, cfg)
            timings["preprocessing"] += time.perf_counter() - t0
            r2 = run_qwen_ocr(region["image"],
                              build_name_prompt(handwritten, "This is part of the form."),
                              cfg.MAX_NEW_TOKENS_TARGETED, engine, cfg=cfg)
            timings["handwriting_inference" if handwritten else "inference"] += r2.inference_s
            calls += 1
            infer += r2.inference_s
            f2, ok2, _ = parse_json(r2.text, NAME_FIELDS)
            if ok2:
                for f in list(missing):
                    if CONF_RANK.get(f2[f]["confidence"], 0) > \
                       CONF_RANK.get(fields[f]["confidence"], 0):
                        fields[f] = f2[f]
                        sources[f] = f"tile_{region['source']}"
                        bboxes[f] = region["bbox"]
                        gens[f] = r2
            missing = _needs_retry(fields, missing)
            if not missing:
                break

    out = {f: field_record(fields[f], page=rendered.page_number, source=sources[f],
                           bbox=bboxes[f], gen=gens[f], name=f) for f in NAME_FIELDS}
    return out, calls, infer


def process_secondary_document(customer_id: str, doc_name: str, pdf_path: Path, engine,
                               expected: Dict[str, Any], handwritten: bool,
                               cfg: Config = CFG) -> Dict[str, Any]:
    """Verify the document visually, then extract surname_latin / given_names_latin from it."""
    t_doc = time.perf_counter()
    timings: Dict[str, float] = defaultdict(float)
    errors: List[Dict[str, Any]] = []
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] = time.perf_counter() - t0

    page_records: List[Dict[str, Any]] = []
    names: Dict[str, Any] = {f: field_record({"value": None, "confidence": "unreadable"}, name=f)
                             for f in NAME_FIELDS}
    title_info, calls = None, 0

    for rendered in pages:
        try:
            t0 = time.perf_counter()
            quality = analyze_page(rendered, cfg)
            processed, pre_meta = preprocess_page(rendered.image, quality, cfg)
            prep = time.perf_counter() - t0
            timings["preprocessing"] += prep
            prec = {"customer_id": customer_id, "document": doc_name,
                    "page_number": rendered.page_number,
                    "image_width": rendered.width, "image_height": rendered.height,
                    "render_time": rendered.render_time_s,
                    "preprocessing_time": round(prep, 3), "page_quality": quality,
                    "preprocessing": pre_meta, "status": "OK", "inference_time": 0.0,
                    "n_model_calls": 0}
            if quality["is_blank"]:
                prec["status"] = "SKIPPED_BLANK"
                page_records.append(prec)
                continue

            if cfg.VERIFY_DOCUMENT_TITLES and title_info is None:
                tc = read_page_title(rendered, processed, pdf_path, pre_meta, engine, timings, cfg)
                title_info = tc
                calls += 1
                prec["n_model_calls"] += 1
                prec["title_text"] = tc["title_text"]

            # Extract names only from the pages of a document we could verify, or when title
            # verification is off. A MISMATCH document is not a source of identity data.
            verdict = classify_title(title_info["title_text"] if title_info else None,
                                     title_info.get("other_headings") if title_info else None,
                                     expected, cfg) if title_info else \
                {"status": "UNCERTAIN", "reason": "no title read"}
            if verdict["status"] != "MISMATCH" and any(
                    n["raw_value"] is None for n in names.values()):
                got, c, infer = extract_names_from_page(rendered, processed, pre_meta, pdf_path,
                                                        engine, timings, handwritten, cfg)
                calls += c
                prec["n_model_calls"] += c
                prec["inference_time"] = round(infer, 3)
                for f in NAME_FIELDS:
                    if names[f]["raw_value"] is None and got[f]["raw_value"] is not None:
                        names[f] = got[f]
            page_records.append(prec)
        except Exception as exc:
            errors.append({"customer_id": customer_id, "document": doc_name,
                           "file": pdf_path.name, "page": rendered.page_number,
                           "stage": "process_secondary_document",
                           "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})
            page_records.append({"customer_id": customer_id, "document": doc_name,
                                 "page_number": rendered.page_number, "status": "PAGE_ERROR",
                                 "error": str(exc), "n_model_calls": 0, "inference_time": 0.0})

    verdict = classify_title(title_info["title_text"] if title_info else None,
                             title_info.get("other_headings") if title_info else None,
                             expected, cfg) if title_info else \
        {"status": "UNCERTAIN", "observed_title": None, "reason": "title not checked"}
    return {
        "document": doc_name, "source_pdf": str(pdf_path), "page_count": len(pages),
        "expected_document": expected.get("label"),
        "detected_document": verdict.get("observed_title"),
        "document_verification_status": verdict["status"], "title_check": verdict,
        "fields": names, "pages": page_records,
        "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
        "n_model_calls": calls, "errors": errors,
        "processing_time_seconds": round(time.perf_counter() - t_doc, 2),
    }


def process_fatca_document(customer_id: str, pdf_path: Path, engine,
                           cfg: Config = CFG) -> Dict[str, Any]:
    """FATCA has TWO components with different titles (section 15).

    Component 1: header "FATCA Identification forms(v1.0)", expected to span two pages.
    Component 2: the FORMULAIRE D'IDENTIFICATION ... LOI FATCA title; its name fields may be
    handwritten, so they get the high-resolution handwriting path."""
    t_doc = time.perf_counter()
    timings: Dict[str, float] = defaultdict(float)
    errors: List[Dict[str, Any]] = []
    t0 = time.perf_counter()
    pages = render_pdf_pages(pdf_path, cfg)
    timings["pdf_render"] = time.perf_counter() - t0

    page_titles, page_records, prepared = [], [], {}
    calls = 0
    for rendered in pages:
        t0 = time.perf_counter()
        quality = analyze_page(rendered, cfg)
        processed, pre_meta = preprocess_page(rendered.image, quality, cfg)
        timings["preprocessing"] += time.perf_counter() - t0
        prepared[rendered.page_number] = (rendered, processed, pre_meta)
        tc = {"title_text": None, "other_headings": None}
        if not quality["is_blank"] and cfg.VERIFY_DOCUMENT_TITLES:
            tc = read_page_title(rendered, processed, pdf_path, pre_meta, engine, timings, cfg)
            calls += 1
        page_titles.append({"page_number": rendered.page_number, **tc})
        page_records.append({"customer_id": customer_id, "document": FATCA_DOC,
                             "page_number": rendered.page_number,
                             "image_width": rendered.width, "image_height": rendered.height,
                             "render_time": rendered.render_time_s,
                             "preprocessing_time": pre_meta.get("preprocess_time_s"),
                             "page_quality": quality, "preprocessing": pre_meta,
                             "title_text": tc.get("title_text"),
                             "status": "SKIPPED_BLANK" if quality["is_blank"] else "OK",
                             "n_model_calls": 1 if not quality["is_blank"] else 0,
                             "inference_time": tc.get("inference_s", 0.0) or 0.0})

    components: Dict[str, Any] = {}
    claimed: set = set()
    # component_2 is evaluated FIRST and claims its pages: its title is the more specific of the
    # two, so where both could match, the specific one wins.
    for key in ("component_2", "component_1"):
        spec = FATCA_COMPONENTS[key]
        hits = []
        for pt in page_titles:
            if pt["page_number"] in claimed:
                continue
            v = classify_title(pt["title_text"], pt.get("other_headings"), spec, cfg)
            if v["status"] == "MATCH":
                hits.append({"page_number": pt["page_number"], **v})
                claimed.add(pt["page_number"])
        status = "MATCH" if hits else (
            "UNCERTAIN" if all(not pt["title_text"] for pt in page_titles) else "MISSING")
        components[key] = {
            "expected_title": spec["label"], "status": status,
            "pages_found": [h["page_number"] for h in hits],
            "n_pages_found": len(hits), "expected_pages": spec["expected_pages"],
            "page_count_matches": (len(hits) == spec["expected_pages"]) if hits else None,
            "observed_titles": [pt["title_text"] for pt in page_titles],
        }
        if hits and len(hits) != spec["expected_pages"]:
            components[key]["flag"] = (f"expected {spec['expected_pages']} page(s), "
                                       f"found {len(hits)}")

    # names come from component 2, and may be handwritten
    names = {f: field_record({"value": None, "confidence": "unreadable"}, name=f)
             for f in NAME_FIELDS}
    target_pages = components["component_2"]["pages_found"] or \
        ([p["page_number"] for p in page_titles] if components["component_2"]["status"] ==
         "UNCERTAIN" else [])
    for pn in target_pages:
        if pn not in prepared:
            continue
        rendered, processed, pre_meta = prepared[pn]
        try:
            got, c, infer = extract_names_from_page(rendered, processed, pre_meta, pdf_path,
                                                    engine, timings, True, cfg)
            calls += c
            for f in NAME_FIELDS:
                if names[f]["raw_value"] is None and got[f]["raw_value"] is not None:
                    names[f] = got[f]
            if all(n["raw_value"] is not None for n in names.values()):
                break
        except Exception as exc:
            errors.append({"customer_id": customer_id, "document": FATCA_DOC,
                           "file": pdf_path.name, "page": pn, "stage": "fatca_handwriting",
                           "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})

    overall = ("MATCH" if all(c["status"] == "MATCH" for c in components.values()) else
               "UNCERTAIN" if any(c["status"] == "UNCERTAIN" for c in components.values())
               else "MISMATCH")
    return {
        "document": FATCA_DOC, "source_pdf": str(pdf_path), "page_count": len(pages),
        "expected_document": "FATCA identification forms (2 components)",
        "detected_document": "; ".join(t for t in
                                       [pt["title_text"] for pt in page_titles] if t) or None,
        "document_verification_status": overall, "components": components,
        "fields": names, "pages": page_records,
        "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
        "n_model_calls": calls, "errors": errors,
        "processing_time_seconds": round(time.perf_counter() - t_doc, 2),
    }


print("CELL 17 ready: process_secondary_document(), process_fatca_document()")

In [ ]:
# =========================================================================
# CELL 18 — CUSTOMER DATABASE  (verification reference ONLY)
# =========================================================================
# Section 5 / 30: the database is never an OCR source. Nothing below writes a database value into
# an extracted field. It can only produce a comparison verdict.
def load_customer_database(path: Optional[Path] = None,
                           cfg: Config = CFG) -> Optional[pd.DataFrame]:
    path = Path(path or cfg.CUSTOMER_DATABASE_PATH) if (path or cfg.CUSTOMER_DATABASE_PATH) else None
    if path is None or not path.exists():
        log.warning("customer database not found (%s): database verification will report "
                    "NOT_AVAILABLE for every customer", path)
        return None
    df = pd.read_csv(path, dtype=str).fillna("")
    if "customer_id" not in df.columns:
        log.error("customer database has no customer_id column; ignoring it")
        return None
    df["customer_id"] = df["customer_id"].astype(str).str.strip()
    df = df.set_index("customer_id")
    log.info("customer database: %d records, columns %s", len(df), list(df.columns))
    return df


def database_record(db: Optional[pd.DataFrame], customer_id: str) -> Optional[Dict[str, str]]:
    if db is None or str(customer_id) not in db.index:
        return None
    row = db.loc[str(customer_id)]
    if isinstance(row, pd.DataFrame):          # duplicate ids: refuse to pick one silently
        log.warning("customer %s appears %d times in the database; skipping comparison",
                    customer_id, len(row))
        return None
    return {k: str(v) for k, v in row.to_dict().items()}


DB = load_customer_database(cfg=CFG)
print("CELL 18 ready: database loaded" if DB is not None else "CELL 18 ready: no database")

In [ ]:
# =========================================================================
# CELL 19 — STAGE B: VERIFICATION AND MATCHING
# =========================================================================
# Statuses (section 32). The critical distinction:
#   NULL is NOT a MISMATCH. Unreadable means UNCERTAIN / NOT_AVAILABLE, never "different".
#   MISMATCH requires two values that were BOTH read and are visibly different.
# Fuzzy scores are always exposed with their threshold, and a fuzzy result is labelled as such --
# never silently promoted to MATCH (section 34).
MATCH, MISMATCH, UNCERTAIN, NOT_AVAILABLE = "MATCH", "MISMATCH", "UNCERTAIN", "NOT_AVAILABLE"


def compare_values(doc_value: Optional[str], ref_value: Optional[str],
                   doc_confidence: str = "unreadable",
                   cfg: Config = CFG) -> Dict[str, Any]:
    """Compare one extracted value against a reference. Normalisation is for comparison only."""
    a, b = normalize_name(doc_value), normalize_name(ref_value)
    out: Dict[str, Any] = {"document_value": doc_value, "reference_value": ref_value,
                           "normalized_document_value": a, "normalized_reference_value": b,
                           "comparison_method": "normalized_exact",
                           "score": None, "threshold": cfg.NAME_FUZZY_THRESHOLD,
                           "document_confidence": doc_confidence}
    if a is None and b is None:
        return {**out, "status": NOT_AVAILABLE, "reason": "neither side has a value"}
    if a is None:
        return {**out, "status": NOT_AVAILABLE,
                "reason": "document value unreadable -- unreadable is not a mismatch"}
    if b is None:
        return {**out, "status": NOT_AVAILABLE, "reason": "no reference value"}
    if a == b:
        status = MATCH if doc_confidence in ("high", "medium") else UNCERTAIN
        return {**out, "status": status, "score": 1.0,
                "reason": "identical after normalisation" if status == MATCH
                else "values agree but the document read is low confidence"}
    score = round(difflib.SequenceMatcher(None, a, b).ratio(), 3)
    out["score"] = score
    if score >= cfg.NAME_FUZZY_THRESHOLD:
        # Explicitly NOT reported as MATCH: a near-identical string is a reason for a human to
        # look, not a confirmation.
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"near match {score} >= {cfg.NAME_FUZZY_THRESHOLD}, not identical"}
    if doc_confidence in ("low", "unreadable"):
        return {**out, "status": UNCERTAIN, "comparison_method": "difflib_ratio",
                "reason": f"values differ ({score}) but the document read is low confidence"}
    return {**out, "status": MISMATCH, "comparison_method": "difflib_ratio",
            "reason": f"values differ, score {score} < {cfg.NAME_FUZZY_THRESHOLD}"}


def _doc_name_fields(doc_result: Optional[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    if not doc_result:
        return {}
    f = doc_result.get("fields", {})
    return {k: f.get(k, {}) for k in NAME_FIELDS}


def build_verification_matrix(documents: Dict[str, Optional[Dict[str, Any]]],
                              cfg: Config = CFG) -> Dict[str, Any]:
    """Cross-document comparison against the identity document (section 31, 35).

    The identity document is the principal reference, but agreement with it never overrides
    uncertainty elsewhere: a document whose own read was unclear stays UNCERTAIN."""
    identity = documents.get(IDENTITY_DOC)
    ref = _doc_name_fields(identity)
    rows, details = [], []
    for doc_name in DOCUMENTS_REQUIRED:
        res = documents.get(doc_name)
        if res is None:
            rows.append({"document": doc_name, "surname_latin": NOT_AVAILABLE,
                         "given_names_latin": NOT_AVAILABLE, "overall": NOT_AVAILABLE,
                         "document_verification_status": "MISSING"})
            continue
        if doc_name == IDENTITY_DOC:
            statuses = {}
            for f in NAME_FIELDS:
                node = ref.get(f, {})
                statuses[f] = ("reference" if node.get("raw_value") else NOT_AVAILABLE)
            rows.append({"document": doc_name, **statuses, "overall": "reference",
                         "document_verification_status":
                             res.get("document_verification_status")})
            continue

        statuses = {}
        for f in NAME_FIELDS:
            node = res.get("fields", {}).get(f, {})
            cmp_ = compare_values(node.get("raw_value"), ref.get(f, {}).get("raw_value"),
                                  node.get("confidence", "unreadable"), cfg)
            statuses[f] = cmp_["status"]
            details.append({"customer_document": doc_name, "field": f,
                            "source_document": doc_name,
                            "reference_source": IDENTITY_DOC, **cmp_})
        overall = (MISMATCH if MISMATCH in statuses.values() else
                   UNCERTAIN if UNCERTAIN in statuses.values() else
                   NOT_AVAILABLE if all(s == NOT_AVAILABLE for s in statuses.values()) else MATCH)
        rows.append({"document": doc_name, **statuses, "overall": overall,
                     "document_verification_status": res.get("document_verification_status")})
    return {"matrix": rows, "comparisons": details}


def verify_against_database(documents: Dict[str, Optional[Dict[str, Any]]],
                            db_row: Optional[Dict[str, str]],
                            cfg: Config = CFG) -> Dict[str, Any]:
    """Compare each document's extracted names against the database record (section 34)."""
    if db_row is None:
        return {"status": NOT_AVAILABLE, "reason": "no database record for this customer",
                "comparisons": []}
    ref = {"surname_latin": db_row.get("surname") or db_row.get("surname_latin"),
           "given_names_latin": db_row.get("given_names") or db_row.get("given_names_latin")}
    full_name = db_row.get("full_name")
    comparisons, statuses = [], []
    for doc_name in DOCUMENTS_REQUIRED:
        res = documents.get(doc_name)
        if not res:
            continue
        for f in NAME_FIELDS:
            node = res.get("fields", {}).get(f, {})
            if node.get("raw_value") is None and ref.get(f) is None:
                continue
            cmp_ = compare_values(node.get("raw_value"), ref.get(f),
                                  node.get("confidence", "unreadable"), cfg)
            cmp_.update({"source_document": doc_name, "field": f, "reference_source": "database"})
            comparisons.append(cmp_)
            statuses.append(cmp_["status"])
    # A full-name fallback, only when the database has no split fields at all.
    if full_name and not any(ref.values()):
        ident = documents.get(IDENTITY_DOC)
        if ident:
            joined = " ".join(x for x in [
                (ident["fields"].get("given_names_latin") or {}).get("raw_value"),
                (ident["fields"].get("surname_latin") or {}).get("raw_value")] if x) or None
            cmp_ = compare_values(joined, full_name,
                                  (ident["fields"].get("surname_latin") or {}).get(
                                      "confidence", "unreadable"), cfg)
            cmp_.update({"source_document": IDENTITY_DOC, "field": "full_name",
                         "reference_source": "database"})
            comparisons.append(cmp_)
            statuses.append(cmp_["status"])

    overall = (MISMATCH if MISMATCH in statuses else
               UNCERTAIN if UNCERTAIN in statuses else
               MATCH if MATCH in statuses else NOT_AVAILABLE)
    return {"status": overall, "database_record": db_row, "comparisons": comparisons}


def overall_kyc_verdict(matrix: Dict[str, Any], db_check: Dict[str, Any],
                        documents: Dict[str, Optional[Dict[str, Any]]]) -> Dict[str, Any]:
    """Final verdict. Anything unresolved routes to review: silence is not approval."""
    reasons = []
    doc_statuses = {r["document"]: r.get("document_verification_status") for r in matrix["matrix"]}
    mismatched_docs = [d for d, s in doc_statuses.items() if s == "MISMATCH"]
    if mismatched_docs:
        reasons.append("document_type_mismatch:" + ",".join(mismatched_docs))
    name_mismatch = [r["document"] for r in matrix["matrix"] if r.get("overall") == MISMATCH]
    if name_mismatch:
        reasons.append("name_mismatch_vs_identity:" + ",".join(name_mismatch))
    if db_check["status"] == MISMATCH:
        reasons.append("name_mismatch_vs_database")

    identity = documents.get(IDENTITY_DOC)
    if not identity:
        return {"overall_kyc_status": "NOT_AVAILABLE",
                "reasons": ["identity document missing"], "needs_manual_review": True}
    crit = ["surname_latin", "given_names_latin", "date_of_birth", "document_number"]
    unread = [f for f in crit if (identity["fields"].get(f) or {}).get("raw_value") is None]
    if unread:
        reasons.append("unreadable_identity_fields:" + ",".join(unread))
    mrz = identity.get("mrz", {})
    if mrz.get("mrz_detected") and not mrz.get("mrz_value"):
        reasons.append("mrz_detected_but_unreadable")
    if identity.get("validation", {}).get("mrz", {}).get("status") == "checksum_mismatch":
        reasons.append("mrz_checksum_mismatch")
    if any(d and d.get("errors") for d in documents.values() if d):
        reasons.append("processing_errors_present")

    if reasons and (mismatched_docs or name_mismatch or db_check["status"] == MISMATCH):
        status = "MISMATCH"
    elif reasons:
        status = "UNCERTAIN"
    elif db_check["status"] == MATCH:
        status = "MATCH"
    else:
        status = "UNCERTAIN"
        reasons.append("no database confirmation available")
    return {"overall_kyc_status": status, "reasons": reasons,
            "needs_manual_review": status != "MATCH"}


print("CELL 19 ready: compare_values(), build_verification_matrix(), overall_kyc_verdict()")

In [ ]:
# =========================================================================
# CELL 20 — CUSTOMER ORCHESTRATION  (stage A then stage B, strictly separated)
# =========================================================================
SECONDARY_SPECS = {
    DOMICILE_DOC: {"expected": EXPECTED_TITLES[DOMICILE_DOC], "handwritten": True},
    CONVENTION_DOC: {"expected": EXPECTED_TITLES[CONVENTION_DOC], "handwritten": True},
    SIGNATURE_DOC: {"expected": EXPECTED_TITLES[SIGNATURE_DOC], "handwritten": True},
}


def process_customer(cust: CustomerDocs, engine, db: Optional[pd.DataFrame] = None,
                     cfg: Config = CFG) -> Dict[str, Any]:
    """One customer, fully independent. Never raises: a failure must not stop the batch."""
    t0 = time.perf_counter()
    documents: Dict[str, Optional[Dict[str, Any]]] = {d: None for d in DOCUMENTS_REQUIRED}
    errors: List[Dict[str, Any]] = []

    # ---------------- STAGE A: visual extraction ----------------
    try:
        if cust.has_identity:
            documents[IDENTITY_DOC] = process_identity_document(
                cust.customer_id, cust.identity_pdf, engine, cfg)
    except Exception as exc:
        errors.append({"customer_id": cust.customer_id, "document": IDENTITY_DOC,
                       "file": str(cust.identity_pdf), "page": None, "stage": "identity_document",
                       "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})
        log.error("identity document failed for %s: %s", cust.customer_id, exc)

    if cfg.PROCESS_SECONDARY_DOCUMENTS:
        for doc_name, spec in SECONDARY_SPECS.items():
            path = cust.documents.get(doc_name)
            if not path:
                continue
            try:
                documents[doc_name] = process_secondary_document(
                    cust.customer_id, doc_name, Path(path), engine,
                    spec["expected"], spec["handwritten"], cfg)
            except Exception as exc:
                errors.append({"customer_id": cust.customer_id, "document": doc_name,
                               "file": path, "page": None, "stage": "secondary_document",
                               "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})
                log.error("%s failed for %s: %s", doc_name, cust.customer_id, exc)
        if cust.documents.get(FATCA_DOC):
            try:
                documents[FATCA_DOC] = process_fatca_document(
                    cust.customer_id, Path(cust.documents[FATCA_DOC]), engine, cfg)
            except Exception as exc:
                errors.append({"customer_id": cust.customer_id, "document": FATCA_DOC,
                               "file": cust.documents[FATCA_DOC], "page": None,
                               "stage": "fatca_document",
                               "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()})
                log.error("FATCA failed for %s: %s", cust.customer_id, exc)

    # ---------------- STAGE B: verification (reads stage A, never writes to it) ----------------
    db_row = database_record(db, cust.customer_id)
    matrix = build_verification_matrix(documents, cfg)
    db_check = verify_against_database(documents, db_row, cfg)
    verdict = overall_kyc_verdict(matrix, db_check, documents)

    # annotate match_status onto the field records WITHOUT touching raw_value
    by_doc_field = defaultdict(dict)
    for c in matrix["comparisons"]:
        by_doc_field[c["source_document"]][c["field"]] = c["status"]
    for doc_name, res in documents.items():
        if not res:
            continue
        for f, node in res.get("fields", {}).items():
            if f in NAME_FIELDS:
                node["match_status"] = by_doc_field.get(doc_name, {}).get(f)

    for res in documents.values():
        if res:
            errors.extend(res.get("errors", []))
    timings: Dict[str, float] = defaultdict(float)
    for res in documents.values():
        if res:
            for k, v in res.get("timings_seconds", {}).items():
                timings[k] += v

    identity = documents.get(IDENTITY_DOC) or {}
    return {
        "customer_id": cust.customer_id,
        "run_id": RUN_ID, "extracted_at_utc": utcnow(),
        "engine": {"model": getattr(engine, "name", "?"), "attn": ATTN_IMPL,
                   "max_new_tokens": cfg.MAX_NEW_TOKENS,
                   "max_image_dimension": cfg.MAX_IMAGE_DIMENSION,
                   "mrz_render_dpi": cfg.MRZ_RENDER_DPI},
        "documents_present": {d: bool(cust.documents.get(d)) for d in DOCUMENTS_REQUIRED},
        "documents": documents,                 # STAGE A output: raw transcriptions
        "identity_fields": identity.get("fields", {}),
        "mrz": identity.get("mrz", {"mrz_detected": False}),
        "verification_matrix": matrix["matrix"],   # STAGE B output: comparisons only
        "cross_document_comparisons": matrix["comparisons"],
        "database_verification": db_check,
        **verdict,
        "page_count_total": sum((d or {}).get("page_count", 0) for d in documents.values()),
        "n_model_calls": sum((d or {}).get("n_model_calls", 0) for d in documents.values()),
        "timings_seconds": {k: round(v, 3) for k, v in timings.items()},
        "processing_time_seconds": round(time.perf_counter() - t0, 2),
        "errors": errors,
    }


print("CELL 20 ready: process_customer()")

In [ ]:
# =========================================================================
# CELL 21 — BATCH PROCESSING
# =========================================================================
def run_batch(targets: List[CustomerDocs], engine=None, db: Optional[pd.DataFrame] = None,
              cfg: Config = CFG) -> pd.DataFrame:
    """Process every customer. One failure never stops the batch (section 21)."""
    engine = engine or ENGINE
    db = db if db is not None else DB
    rows, consecutive_errors = [], 0
    t_start = time.perf_counter()

    for i, cust in enumerate(targets, 1):
        dest = DIRS["results"] / f"{cust.customer_id}.json"
        if cfg.RESUME and dest.exists():
            log.info("[%d/%d] %s skipped (already done)", i, len(targets), cust.customer_id)
            continue
        try:
            rec = process_customer(cust, engine, db, cfg)
        except Exception as exc:                    # defence in depth
            log.error("[%d/%d] %s FAILED: %s", i, len(targets), cust.customer_id, exc)
            (DIRS["logs"] / f"{cust.customer_id}.error.txt").write_text(
                traceback.format_exc(), encoding="utf-8")
            rec = {"customer_id": cust.customer_id, "overall_kyc_status": "ERROR",
                   "needs_manual_review": True, "reasons": [str(exc)], "documents": {},
                   "identity_fields": {}, "verification_matrix": [], "n_model_calls": 0,
                   "processing_time_seconds": 0.0, "page_count_total": 0,
                   "timings_seconds": {},
                   "errors": [{"customer_id": cust.customer_id, "document": None, "file": None,
                               "page": None, "stage": "process_customer",
                               "error": f"{type(exc).__name__}: {exc}", "timestamp": utcnow()}]}
        dest.write_text(json.dumps(rec, ensure_ascii=False, indent=2, default=str),
                        encoding="utf-8")

        mrz = rec.get("mrz") or {}
        log.info("[%d/%d] %s %s | %dp %.1fs calls=%d | mrz=%s | db=%s",
                 i, len(targets), cust.customer_id, rec.get("overall_kyc_status"),
                 rec.get("page_count_total", 0), rec.get("processing_time_seconds", 0),
                 rec.get("n_model_calls", 0),
                 "ok" if mrz.get("mrz_value") else mrz.get("mrz_validation_status") or "none",
                 (rec.get("database_verification") or {}).get("status"))
        rows.append({"customer_id": cust.customer_id,
                     "overall_kyc_status": rec.get("overall_kyc_status"),
                     "pages": rec.get("page_count_total", 0),
                     "time_s": rec.get("processing_time_seconds"),
                     "model_calls": rec.get("n_model_calls"),
                     "mrz_value_read": bool(mrz.get("mrz_value")),
                     "mrz_page": mrz.get("mrz_page"),
                     "database_status": (rec.get("database_verification") or {}).get("status"),
                     "needs_manual_review": rec.get("needs_manual_review")})

        if rec.get("overall_kyc_status") == "ERROR":
            consecutive_errors += 1
            if consecutive_errors >= cfg.MAX_CONSECUTIVE_ERRORS:
                log.error("ABORTING BATCH after %d consecutive system errors. The documents are "
                          "not the problem. Run gpu_report(); free_model() or restart the kernel; "
                          "raise CFG.RESERVE_VRAM_GIB or lower CFG.MAX_VISUAL_TOKENS.",
                          consecutive_errors)
                break
        else:
            consecutive_errors = 0

    log.info("batch finished in %.1f min", (time.perf_counter() - t_start) / 60)
    return pd.DataFrame(rows)


print("CELL 21 ready: run_batch()")

In [ ]:
# =========================================================================
# CELL 22 — PERFORMANCE LOGGING AND OUTPUT GENERATION
# =========================================================================
def load_results(results_dir: Path = None) -> List[Dict[str, Any]]:
    out = []
    for p in sorted(Path(results_dir or DIRS["results"]).glob("*.json")):
        try:
            out.append(json.loads(p.read_text(encoding="utf-8")))
        except Exception as exc:
            log.warning("unreadable result %s: %s", p.name, exc)
    return out


def build_processing_log(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """One row per PAGE across all documents (section 10)."""
    rows = []
    for r in records:
        for doc_name, res in (r.get("documents") or {}).items():
            if not res:
                continue
            for p in res.get("pages", []):
                mrz = p.get("mrz_report") or {}
                rows.append({
                    "customer_id": r["customer_id"], "document": doc_name,
                    "page_number": p.get("page_number"),
                    "image_width": p.get("image_width"), "image_height": p.get("image_height"),
                    "render_time": p.get("render_time"),
                    "preprocessing_time": p.get("preprocessing_time"),
                    "inference_time": p.get("inference_time"),
                    "mrz_inference_time": mrz.get("mrz_inference_time"),
                    "n_model_calls": p.get("n_model_calls"),
                    "quality": (p.get("page_quality") or {}).get("quality"),
                    "text_height_px": (p.get("page_quality") or {}).get("text_height_px"),
                    "rotation": (p.get("preprocessing") or {}).get("rotation"),
                    "mrz_detected": p.get("mrz_detected"),
                    "mrz_source": mrz.get("mrz_source"),
                    "title_text": p.get("title_text"),
                    "status": p.get("status"),
                })
    return pd.DataFrame(rows)


def flatten_customer(r: Dict[str, Any]) -> Dict[str, Any]:
    row: Dict[str, Any] = OrderedDict(customer_id=r["customer_id"],
                                      overall_kyc_status=r.get("overall_kyc_status"),
                                      needs_manual_review=r.get("needs_manual_review"),
                                      reasons=";".join(r.get("reasons") or []),
                                      page_count_total=r.get("page_count_total"),
                                      processing_time_seconds=r.get("processing_time_seconds"),
                                      n_model_calls=r.get("n_model_calls"))
    for doc, present in (r.get("documents_present") or {}).items():
        row[PRESENCE_COLUMNS.get(doc, doc)] = "TRUE" if present else "FALSE"
    ident = (r.get("documents") or {}).get(IDENTITY_DOC) or {}
    for f in IDENTITY_FIELDS:
        n = (ident.get("fields") or {}).get(f) or {}
        row[f] = n.get("raw_value")
        row[f + "_normalized"] = n.get("normalized_value")
        row[f + "_confidence"] = n.get("confidence")
        row[f + "_page"] = n.get("page_number")
    mrz = r.get("mrz") or {}
    row.update({"mrz_detected": mrz.get("mrz_detected"), "mrz_page": mrz.get("mrz_page"),
                "mrz_crop_width": mrz.get("mrz_crop_width"),
                "mrz_crop_height": mrz.get("mrz_crop_height"),
                "mrz_inference_time": mrz.get("mrz_inference_time"),
                "mrz_validation_status": mrz.get("mrz_validation_status"),
                "mrz_source": mrz.get("mrz_source")})
    for doc_name in DOCUMENTS_REQUIRED:
        res = (r.get("documents") or {}).get(doc_name)
        key = PRESENCE_COLUMNS.get(doc_name, doc_name).replace("_exists", "")
        row[f"{key}_verification"] = (res or {}).get("document_verification_status", "MISSING")
        if doc_name != IDENTITY_DOC:
            for f in NAME_FIELDS:
                n = ((res or {}).get("fields") or {}).get(f) or {}
                row[f"{key}_{f}"] = n.get("raw_value")
                row[f"{key}_{f}_status"] = n.get("match_status")
    row["database_status"] = (r.get("database_verification") or {}).get("status")
    return row


def write_outputs(records: List[Dict[str, Any]] = None, presence_df=None,
                  cfg: Config = CFG) -> Dict[str, Path]:
    records = records if records is not None else load_results()
    rep, paths = DIRS["reports"], {}

    with (rep / "identity_extraction_results.jsonl").open("w", encoding="utf-8") as fh:
        for r in records:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
    paths["jsonl"] = rep / "identity_extraction_results.jsonl"

    flat = pd.DataFrame([flatten_customer(r) for r in records]) if records else pd.DataFrame()
    flat.to_csv(rep / "identity_extraction_results.csv", index=False, encoding="utf-8-sig")
    paths["csv"] = rep / "identity_extraction_results.csv"

    matrix_rows = [{"customer_id": r["customer_id"], **m}
                   for r in records for m in (r.get("verification_matrix") or [])]
    pd.DataFrame(matrix_rows).to_csv(rep / "verification_matrix.csv", index=False,
                                     encoding="utf-8-sig")
    paths["verification_matrix"] = rep / "verification_matrix.csv"

    cmp_rows = [{"customer_id": r["customer_id"], **c} for r in records
                for c in (r.get("cross_document_comparisons") or [])] + \
               [{"customer_id": r["customer_id"], **c} for r in records
                for c in ((r.get("database_verification") or {}).get("comparisons") or [])]
    pd.DataFrame(cmp_rows).to_csv(rep / "verification_comparisons.csv", index=False,
                                  encoding="utf-8-sig")
    paths["comparisons"] = rep / "verification_comparisons.csv"

    build_processing_log(records).to_csv(rep / "processing_log.csv", index=False,
                                         encoding="utf-8-sig")
    paths["processing_log"] = rep / "processing_log.csv"

    err_rows = [e for r in records for e in (r.get("errors") or [])]
    pd.DataFrame(err_rows, columns=["customer_id", "document", "file", "page", "stage",
                                    "error", "timestamp"]).to_csv(
        rep / "errors.csv", index=False, encoding="utf-8-sig")
    paths["errors"] = rep / "errors.csv"

    for k, v in paths.items():
        print(f"  {k:<22}: {v}")
    if len(flat):
        print("\nKYC status:", dict(flat["overall_kyc_status"].value_counts()))
    return paths


def benchmark_pipeline(records: List[Dict[str, Any]] = None, cfg: Config = CFG) -> Dict[str, Any]:
    """Mean / median / p95 per stage: identifies the bottleneck instead of assuming it."""
    records = records if records is not None else load_results()
    if not records:
        print("no results yet")
        return {}

    def st(v):
        a = np.asarray([x for x in v if x is not None], dtype=float)
        return {"mean": None, "median": None, "p95": None} if a.size == 0 else {
            "mean": round(float(a.mean()), 2), "median": round(float(np.median(a)), 2),
            "p95": round(float(np.percentile(a, 95)), 2)}

    tim = [r.get("timings_seconds", {}) for r in records]
    keys = ["pdf_render", "preprocessing", "inference", "mrz_inference",
            "title_inference", "handwriting_inference", "json_parse"]
    per_stage = {f"{k}_s": st([t.get(k) for t in tim]) for k in keys}
    per_stage["total_s"] = st([r.get("processing_time_seconds") for r in records])
    totals = {k: sum(t.get(k, 0) or 0 for t in tim) for k in keys}
    grand = sum(totals.values()) or 1.0

    mrzs = [r.get("mrz") or {} for r in records]
    detected = [m for m in mrzs if m.get("mrz_detected")]
    read = [m for m in detected if m.get("mrz_value")]
    out = {
        "n_customers": len(records),
        "per_stage": per_stage,
        "share_of_measured_time_pct": {k: round(100 * v / grand, 1)
                                       for k, v in sorted(totals.items(), key=lambda x: -x[1])},
        "avg_model_calls_per_customer": round(float(np.mean(
            [r.get("n_model_calls", 0) for r in records])), 2),
        "mrz_detected": len(detected), "mrz_read": len(read),
        "mrz_read_rate": round(len(read) / max(1, len(detected)), 3),
        "mrz_sources": dict(pd.Series([m.get("mrz_source") for m in detected]
                                      ).value_counts(dropna=False)),
        "kyc_status": dict(pd.Series([r.get("overall_kyc_status") for r in records]
                                     ).value_counts()),
    }
    print("=" * 68)
    print(f"customers {out['n_customers']}   calls/customer {out['avg_model_calls_per_customer']}")
    print(f"{'stage':<26}{'mean':>10}{'median':>10}{'p95':>10}")
    for k, v in per_stage.items():
        print(f"{k:<26}{str(v['mean']):>10}{str(v['median']):>10}{str(v['p95']):>10}")
    print("-" * 68)
    print("share of measured time:", out["share_of_measured_time_pct"])
    print(f"MRZ: detected {out['mrz_detected']}, read {out['mrz_read']} "
          f"({out['mrz_read_rate']}), sources {out['mrz_sources']}")
    print("KYC:", out["kyc_status"])
    print("=" * 68)
    (DIRS["reports"] / "benchmark_summary.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return out


print("CELL 22 ready: write_outputs(), benchmark_pipeline()")

In [ ]:
# =========================================================================
# CELL 23 — RESULT INSPECTION (audit trail view)
# =========================================================================
def inspect_customer(customer_id: str) -> Optional[Dict[str, Any]]:
    path = DIRS["results"] / f"{customer_id}.json"
    if not path.exists():
        print("no result for", customer_id)
        return None
    r = json.loads(path.read_text(encoding="utf-8"))
    print("=" * 78)
    print(f"{r['customer_id']}   KYC: {r.get('overall_kyc_status')}   "
          f"review: {r.get('needs_manual_review')}   {r.get('processing_time_seconds')}s   "
          f"calls={r.get('n_model_calls')}")
    if r.get("reasons"):
        print("reasons:", "; ".join(r["reasons"]))

    print("-" * 78)
    print("STAGE A - identity document (raw transcription)")
    ident = (r.get("documents") or {}).get(IDENTITY_DOC) or {}
    print(f"  verification: {ident.get('document_verification_status')} "
          f"| detected title: {ident.get('detected_document')}")
    for f in IDENTITY_FIELDS:
        n = (ident.get("fields") or {}).get(f) or {}
        v = n.get("raw_value")
        v = (v[:44] + "...") if isinstance(v, str) and len(v) > 47 else v
        print(f"  {f:<20} {str(v):<48} {n.get('confidence','-'):<11} "
              f"p{n.get('page_number','-')} {n.get('source_region') or ''}")

    mrz = r.get("mrz") or {}
    print("-" * 78)
    print("MRZ pipeline:")
    for k in ("mrz_detected", "mrz_page", "mrz_crop_width", "mrz_crop_height",
              "mrz_inference_time", "mrz_validation_status", "mrz_source"):
        print(f"  {k:<24} {mrz.get(k)}")
    for a in mrz.get("attempts", []):
        print(f"    attempt {a['attempt']}: {a['source']:<20} crop={a['crop']} "
              f"char~{a['estimated_char_height_px']}px tokens={a['visual_tokens']} "
              f"-> {'value' if a['got_value'] else 'null'} ({a['validation_status']})")
    for k, c in (mrz.get("mrz_validation", {}) or {}).get("checks", {}).items():
        mark = "ok" if c.get("computed") == c.get("printed") else "MISMATCH"
        print(f"    check {k:<16} computed={c.get('computed')} printed={c.get('printed')} {mark}")

    print("-" * 78)
    print("STAGE A - secondary documents")
    for doc in DOCUMENTS_REQUIRED[1:]:
        res = (r.get("documents") or {}).get(doc)
        if not res:
            print(f"  {doc:<28} MISSING")
            continue
        print(f"  {doc:<28} {res.get('document_verification_status'):<10} "
              f"title={str(res.get('detected_document'))[:40]}")
        for f in NAME_FIELDS:
            n = (res.get("fields") or {}).get(f) or {}
            print(f"      {f:<18} {str(n.get('raw_value')):<28} {n.get('confidence','-'):<11} "
                  f"{n.get('text_type') or '-':<12} {n.get('match_status') or '-'}")
        if res.get("components"):
            for k, c in res["components"].items():
                print(f"      {k:<18} {c['status']:<10} pages={c['pages_found']} "
                      f"expected={c['expected_pages']} {c.get('flag','')}")

    print("-" * 78)
    print("STAGE B - verification matrix")
    print(f"  {'document':<28}{'surname':<16}{'given names':<16}{'overall':<16}doc_type")
    for m in r.get("verification_matrix", []):
        print(f"  {m['document']:<28}{str(m.get('surname_latin')):<16}"
              f"{str(m.get('given_names_latin')):<16}{str(m.get('overall')):<16}"
              f"{m.get('document_verification_status')}")
    db = r.get("database_verification") or {}
    print(f"  DATABASE VERIFICATION: {db.get('status')}")
    for c in (db.get("comparisons") or [])[:8]:
        print(f"    {c['source_document'][:22]:<24}{c['field']:<18}"
              f"doc={str(c['document_value'])[:18]:<20}db={str(c['reference_value'])[:18]:<20}"
              f"{c['status']} ({c['comparison_method']} score={c['score']})")
    print(f"  OVERALL KYC VERIFICATION: {r.get('overall_kyc_status')}")
    print("timings:", r.get("timings_seconds"))
    return r


print("CELL 23 ready: inspect_customer('CUSTOMER_ID')")

## Driver

In [ ]:
# =========================================================================
# CELL 24 — DRIVER
# =========================================================================
catalog_df = extract_zip(CFG.zip_path, CFG)
customers = discover_customers(catalog_df)
presence_df = build_presence_report(customers, catalog_df, CFG)
targets = select_identity_targets(customers, CFG)

batch_df = run_batch(targets, ENGINE, DB, CFG)
records = load_results()
paths = write_outputs(records, presence_df, CFG)
summary = benchmark_pipeline(records, CFG)

if len(batch_df):
    display(batch_df)
    inspect_customer(batch_df.iloc[0]["customer_id"])

## Self-audit and known limits

**Preserved from the previous notebook, not replaced.** Same `AutoProcessor`, same architecture
lookup from `config.json["architectures"]` with `AutoModelForImageTextToText` /
`AutoModelForVision2Seq` fallbacks, same `apply_chat_template` with
`[{"type": "image"}, {"type": "text"}]` content, same `processor(text=[...], images=[...])` call,
same `model.generate` under `inference_mode`. What changed is how it is *driven*: token budget,
stop condition, thinking disabled, region re-rendering, and one model/processor per kernel.

| Requirement | Where |
|---|---|
| Model + processor loaded once | CELL 5 singleton; `free_model()` to release |
| Every page rendered and recorded | CELL 9 + per-page records in CELLS 16/17 |
| MRZ dedicated path, page 2 | CELL 11 detection, CELL 16 `process_mrz` |
| MRZ validation cannot invent | CELL 14 `validate_mrz`, report-only; mismatch never retries |
| Latin names only, no transliteration | CELL 12b prompt + CELL 14 `non_latin_characters_in_latin_field` |
| Handwriting path | CELL 17 `extract_names_from_page` with PDF-clip tiles |
| Document classification | CELL 15: model transcribes, Python decides |
| FATCA two components | CELL 17 `process_fatca_document` |
| Raw vs normalized | CELL 13 `field_record`: `raw_value` never overwritten |
| NULL ≠ MISMATCH | CELL 19 `compare_values` returns NOT_AVAILABLE for unreadable |
| Database never completes a value | CELL 18/19: comparison only, no write path into stage A |
| Fuzzy scores exposed | CELL 19: score + threshold on every comparison |
| Audit trail | CELL 13 `field_record`: page, region, bbox, confidence, token confidence |
| Errors don't stop the batch | CELLS 17, 20, 21 + `errors.csv` |
| Flash Attention detected | CELL 4 `detect_attention_impl()`, falls back to sdpa/eager |
| Deterministic generation | CELL 12: greedy, `repetition_penalty=1.0`, `no_repeat_ngram_size=0` |

### Deliberate design choices worth knowing about

**A near-identical name returns UNCERTAIN, not MATCH.** `compare_values` only reports MATCH on an
exact match after normalisation. A difflib ratio of 0.93 between `MOHAMED` and `MOHAMMED` is a
reason for a human to look, not a confirmation — and section 34 forbids silently converting a
fuzzy match into an exact one. Raise `NAME_FUZZY_THRESHOLD` toward 1.0 to narrow the UNCERTAIN
band, but it will never auto-approve.

**Title verification asks for a transcription, not a yes/no.** Asking a VLM "is this the signature
card?" reliably gets "yes". The model transcribes the heading and Python compares, so the verdict
is deterministic and the observed title is in the audit trail either way.

**`SPICIMEN` vs `SPÉCIMEN`.** Both spellings match, and `detected_document` records what was
actually read. If your corpus genuinely contains the typo, you will see it in
`verification_matrix.csv` rather than having it normalised away.

### Things to measure rather than trust

- `MAX_IMAGE_DIMENSION = 1400` is a starting point. The MRZ no longer depends on it (it has its
  own 600 dpi clip), but printed field legibility does. Test 1280 / 1400 / 1600 on a labelled
  sample before settling.
- Handwriting extraction uses page halves, not located fields. On forms where the name sits in the
  lower third, the second tile catches it; on dense multi-column forms you may need
  `HANDWRITING_TILES = 3`. The tile that produced each value is recorded in `source_region`.
- The 180° orientation heuristic is the weakest component — it reasons about ink position relative
  to a text baseline, which is weaker on sparse cards than on dense forms. Install tesseract so
  the OSD tier takes over if `preprocessing.orientation.method` is mostly `cv_projection`.
- **Your spec is truncated mid-section 37** (the identity output schema). The schema here follows
  section 33: every field carries `raw_value`, `normalized_value`, `confidence`, `match_status`,
  plus page, region and bbox for the audit trail. If section 37 specified something different,
  that is the one place to reconcile.